# 2 stages optimization tool

## 1. Importing libraries and dataset

In [ ]:
import numpy as np
import math
import os

# Install pyarrow if not already installed
import subprocess
import sys

try:
    import pyarrow
    print("pyarrow is already installed")
except ImportError:
    print("Installing pyarrow...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--user", "pyarrow", "-q"])
    print("pyarrow installed successfully!")
    print("NOTE: If you encounter extension type errors, restart the kernel (Kernel -> Restart)")

### 1.1 Importing TokWise´s parquet data, and aFRR data.

--> Check readme file ans use in input the file of interest

In [ ]:
import pandas as pd
from pathlib import Path
import pyarrow.parquet as pq

parquet_path = Path.cwd() / "filtered_solar_data_augmented_realistic.parquet"  # same folder as this notebook when run from its directory

# Workaround for pyarrow extension type conflict: use pyarrow directly then convert to pandas
# This avoids pandas' extension type registration that causes conflicts
table = pq.read_table(parquet_path)
df = table.to_pandas()

# Rename columns to match what the notebook expects
df = df.rename(columns={
    "DayAheadPrice":                             "DA",
    "IntradayVolumeWeightedAveragePrice":        "closest_VWAP",
    "ImbalanceSettlementPrice":                  "ISP",
    "IntradayForecastCreationtLeadTimeMinutes":  "lead_time",
    "Capacity_kWh":                              "Capacity",
})

# Convert kWh columns to MWh so units match the rest of the notebook
df["fc_mwh_curr"] = df["Intraday_Forecast_kWh"] / 1000
df["act_mwh"]     = df["Generation_kWh"] / 1000


# Diagnostic: Check date range in raw data
print(f"Raw dataset info:")
print(f"  Total rows: {len(df)}")
if 'PeriodStartUTC' in df.columns:
    print(f"  Date range (UTC): {df['PeriodStartUTC'].min()} to {df['PeriodStartUTC'].max()}")


In [ ]:
print(df.groupby('SolarId').agg(
    Capacity=('Capacity', 'first'),
    Rows=('PeriodStartUTC', 'count'),
    Max_Generation_kWh=('Generation_kWh', 'max')
).sort_values('Capacity', ascending=False))

In [ ]:
df_full = pd.read_parquet('full_dataset_ino_energy_final.parquet')
print(df_full.columns.tolist())
print(df_full.groupby('SolarId').agg(
    Capacity=('Capacity_kWh', 'first'),
    Rows=('PeriodStartUTC', 'count')
).sort_values('Capacity', ascending=False).head(10))

In [ ]:
df_full = pd.read_parquet('full_dataset_ino_energy_final.parquet')
print(df_full.groupby('SolarId').agg(
    Capacity_kW=('Capacity_kWh', 'first'),
    Missing_Generation=('Generation_kWh', lambda x: x.isna().sum()),
    Missing_Forecast=('Intraday_Forecast_kWh', lambda x: x.isna().sum()),
    Total_Rows=('PeriodStartUTC', 'count')
).sort_values('Capacity_kW', ascending=False))

In [ ]:
print(df.columns.tolist())

aFRR data

In [ ]:
import os
# Define the folder name
folder = "Market_data"

BCap_p = pd.read_excel(os.path.join(folder, "RESULT_OVERVIEW_CAPACITY_MARKET_aFRR_2025-10-05_2025-10-05.xlsx"))
EAM_price_data = pd.read_parquet(os.path.join(folder,"aFRR_EAM_optimal_bids_2025_10_05"))

Convert aFRR data to standard format

In [ ]:
# 1. Extract the direction (POS/NEG) and the Hour from Column 3
# Column 3 (index 3) format: "POS_00_04"
direction = BCap_p.iloc[:, 3].str[:3]      # Gets "POS" or "NEG"
hour_str = BCap_p.iloc[:, 3].str[4:6]       # Gets "00"

# 3. Create the combined datetime
# We take the date from Column 0 and add the extracted hour
BCap_p['PeriodStartLocal'] = pd.to_datetime(
    BCap_p.iloc[:, 0].astype(str) + ' ' + hour_str + ':00:00'
).dt.tz_localize("Europe/Berlin")

# 4. Get the numerical value from Column J (Index 9), which is the marginal price of activation [(EUR/MWh)/h] i.e. the highest priced bid cleared.
BCap_p['BCap_price'] = pd.to_numeric(BCap_p.iloc[:, 9], errors='coerce')

# 5. Split into two DataFrames based on the direction
BCap_Pos_p = BCap_p[direction == "POS"][['PeriodStartLocal', 'BCap_price']].copy()
BCap_Neg_p = BCap_p[direction == "NEG"][['PeriodStartLocal', 'BCap_price']].copy()

# 6. Clean up indices
BCap_Pos_p.reset_index(drop=True, inplace=True)
BCap_Neg_p.reset_index(drop=True, inplace=True)

print("Negative Capacity Head:")
BCap_Neg_p.head(6)

Convert BCap data to quarterly data, for easier model handling, and latter the constraints will be set so bid periods act as blocks of 4 hours.

In [ ]:
import pandas as pd

# Ensure PeriodStartLocal is datetime and tz-aware
BCap_Neg_p["PeriodStartLocal"] = pd.to_datetime(BCap_Neg_p["PeriodStartLocal"])
if BCap_Neg_p["PeriodStartLocal"].dt.tz is None:
    BCap_Neg_p["PeriodStartLocal"] = BCap_Neg_p["PeriodStartLocal"].dt.tz_localize("Europe/Berlin")

# Determine the day (date) for the data
day_start = BCap_Neg_p["PeriodStartLocal"].dt.normalize().iloc[0]
day_end = day_start + pd.Timedelta(days=1)  # next day

# Create full quarter-hourly index for the entire day (96 periods)
full_index = pd.date_range(
    start=day_start,
    end=day_end,
    freq="15T",
    tz="Europe/Berlin",
    inclusive="left"  # replace closed='left'
)

# Set original data index
BCap_Neg_p = BCap_Neg_p.set_index("PeriodStartLocal")

# Reindex to full day and forward-fill
BCap_Neg_p = BCap_Neg_p.reindex(full_index).ffill().reset_index()
BCap_Neg_p.rename(columns={"index": "PeriodStartLocal"}, inplace=True)

BCap_Neg_p.tail()

In [ ]:
EAM_price_data.head()

In [ ]:
import pandas as pd

# Copy to avoid modifying original
EAM_p = EAM_price_data.copy()

# Ensure Date is datetime
EAM_p["Date"] = pd.to_datetime(EAM_p["Date"])

# Convert period (1–96) → minutes offset
EAM_p["PeriodStartLocal"] = (
    EAM_p["Date"]
    + pd.to_timedelta((EAM_p["period"] - 1) * 15, unit="m")
)

# Localize to Berlin timezone (handles CET/CEST automatically)
EAM_p["PeriodStartLocal"] = (
    EAM_p["PeriodStartLocal"]
    .dt.tz_localize("Europe/Berlin", ambiguous="infer", nonexistent="shift_forward")
)

# Select + rename columns
EAM_p = EAM_p[[
    "PeriodStartLocal",
    "expected_price_term_eur_per_mwh",
    "expected_activated_energy_mwh"
]].rename(columns={
    "expected_price_term_eur_per_mwh": "EAM_price"
})

In [ ]:
EAM_p.head()

### 1.2 Reading and understanding the dataset

In [ ]:
# Display first 10 rows
df.head()

In [ ]:
df.tail()

In [ ]:
df.dtypes

In [ ]:
df.describe()

## 2. Data cleaning and prices extrapolation

In [ ]:
# Ensure PeriodStartUTC is UTC-aware
df["PeriodStartUTC"] = pd.to_datetime(df["PeriodStartUTC"], utc=True)

# Convert to local time and store in new column
df["PeriodStartLocal"] = df["PeriodStartUTC"].dt.tz_convert("Europe/Berlin")

df.head()

In [ ]:
DA_p = (
    df[["PeriodStartLocal", "DA"]]
      .drop_duplicates(subset=["PeriodStartLocal"])
      .rename(columns={"DA": "DA_price"})
)

DA_p.head()


In [ ]:
# Use closest_VWAP as the latest intraday market price observed
# before the delivery interval, representing the market’s
# expectation of the intraday price at the decision time

ID_p = df.copy()

ID_p = (
    ID_p[["PeriodStartLocal", "closest_VWAP", "lead_time"]]
      .sort_values(["PeriodStartLocal", "lead_time"])          # closest first
      .drop_duplicates(subset=["PeriodStartLocal"], keep="first")
      .rename(columns={
                       "closest_VWAP": "ID_price"})[["PeriodStartLocal", "ID_price"]]
    .sort_values("PeriodStartLocal")
      .reset_index(drop=True)
)

# Fill any NaN ID prices with zero
ID_p["ID_price"] = ID_p["ID_price"].fillna(0)


ID_p.head()

In [ ]:
#Load the imbalance prices, not known until after the period is over.
Imb_p = (
    df[["PeriodStartLocal", "ISP"]]
      .drop_duplicates(subset=["PeriodStartLocal"])
      .rename(columns={"ISP": "Imb_price"})
)

Imb_p.head()

In [ ]:
Imb_p.tail()

Below is the calculation of the EEG subsidies, based on publicly available data on the market premium (https://www.bundesnetzagentur.de/DE/Fachthemen/ElektrizitaetundGas/ErneuerbareEnergien/EEG_Foerderung/start.html ), for which the average value of all 2025 bidding rounds is taken, and on the monthly market values (https://www.netztransparenz.de/en/Renewable-energies-and-levies/EEG/Transparency-requirements/Market-premium/Market-value-overview ). Whilst the market values are published after trading, for simplicity we assume that these values are known to quantify the expected subsidy. In reality, this could be possible through forecasting tools, or simply replacing the actual market value by a realistic assumption.

In [ ]:
# Market values in EUR/MWh by month (example values)
market_values_2025 = {
    1: 115.11,   # January
    2: 110.99,   # February
    3: 50.27,
    4: 30.41,
    5: 19.97,
    6: 18.43,
    7: 59.23,
    8: 38.32,
    9: 43.07,
    10: 69.8,  # October
    11: 91.02,
    12: 93.73
}

reference_value_2025 = 0.1016 * 1000,  # 101.6 EUR/MWh

In [ ]:
def add_market_premium(DA_p: pd.DataFrame,
                       periods_per_hour: int,
                       reference_value: float,
                       market_values: dict) -> pd.DataFrame:
    """
    Adds a 'market premium' column using monthly market values.

    Parameters
    ----------
    DA_p : pd.DataFrame
        Must contain:
        - 'PeriodStartLocal'
        - 'DA_price'
    periods_per_hour : int
        Min number of consecutive periods from which the EEG subsidy 
        does not apply. (Currently 1 period, which is 15 min)
    reference_value : float
        Reference EEG value (EUR/MWh)
    market_values : dict
        Dictionary mapping month (1-12) to market value (EUR/MWh)

    Returns
    -------
    pd.DataFrame
    """

    df = DA_p.copy()

    # Ensure datetime
    df["PeriodStartLocal"] = pd.to_datetime(df["PeriodStartLocal"])

    # Sort chronologically
    df = df.sort_values("PeriodStartLocal").reset_index(drop=True)

    # Extract month
    df["month"] = df["PeriodStartLocal"].dt.month

    # Map monthly market values
    df["market_value"] = df["month"].map(market_values)

    # Calculate monthly premium
    df["market premium"] = np.maximum(reference_value - df["market_value"], 0)

    # Detect negative price periods
    negative = df["DA_price"] < 0

    # Identify blocks of consecutive negatives
    group = (negative != negative.shift()).cumsum()
    block_sizes = negative.groupby(group).transform("sum")

    # Set premium to zero where consecutive negatives >= periods_per_hour
    df.loc[(negative) & (block_sizes >= periods_per_hour),
           "market premium"] = 0

    # Clean helper columns
    df = df.drop(columns=["DA_price", "month", "market_value"])

    return df

In [ ]:
EEG_p = add_market_premium(
    DA_p=DA_p,
    periods_per_hour=1,
    reference_value = reference_value_2025,
    market_values=market_values_2025
)

EEG_p.head()

### 2.1 DA Price Forecast: Previous Day's Prices (Naive Persistence Forecast)

In [ ]:
def generate_day_lag_price_forecast(df, lag_periods=96):
    """
    Creates price forecasts using previous day's actual prices (naive persistence).
    
    IMPORTANT: Input dataframe must be sorted chronologically for shift() to work correctly!
    
    Parameters:
    - df: DataFrame with columns [timestamp, price_column] - MUST BE SORTED BY TIMESTAMP
    - lag_periods: Number of periods to shift (96 = 1 day for quarter-hourly data)
    
    Returns:
    - DataFrame with actual prices and lagged forecasts
    """
    df_new = df.copy()
    target_col_name = df.columns[1]
    
    # Shift prices by 1 day (96 periods) to create forecast
    # For the first day, use the actual price (no previous day available)
    df_new[f"{target_col_name}_Forecast_1"] = df_new[target_col_name].shift(lag_periods)
    
    # Fill NaN values in the first day with the actual prices (fallback)
    df_new[f"{target_col_name}_Forecast_1"] = df_new[f"{target_col_name}_Forecast_1"].fillna(df_new[target_col_name])
    
    # Create Forecast_2 as duplicate of Forecast_1 (same persistence forecast)
    df_new[f"{target_col_name}_Forecast_2"] = df_new[f"{target_col_name}_Forecast_1"]
    
    # Return forecast columns
    cols_to_return = [f"{target_col_name}_Forecast_1", f"{target_col_name}_Forecast_2"]
    return df_new[[df.columns[0], target_col_name] + cols_to_return]

print("DA Price Forecast Method: Using previous day's actual prices (lag = 96 periods)")

In [ ]:
# NOTE: Forecast generation has been moved to Section 4 (before filtering)
# This ensures forecasts are generated from the full dataset including previous day's data
# The code below is kept for reference but not executed

# # Generate DA forecast using previous day's prices
# DA_p_forecasts = generate_day_lag_price_forecast(DA_p, lag_periods=96)

# For ID prices, keep synthetic forecasts (or use same method)
def generate_synthetic_price_forecasts(df, phi=0.9, base_error_scale=0.05):
    """
    Creates two synthetic forecasts using AR(1) noise and variance scaling.
    For ID market prices (kept for comparison).
    """
    df_new = df.copy()
    target_col_name = df.columns[1]
    actual_values = df.iloc[:, 1].values
    n = len(actual_values)

    sample_val = str(actual_values[0])
    if '.' in sample_val:
        precision = len(sample_val.split('.')[1])
    else:
        precision = 0
    
    def create_ar_noise(intensity):
        noise = np.zeros(n)
        innovation = np.random.normal(0, intensity, n) * (actual_values + 1.0)
        noise[0] = innovation[0]
        for t in range(1, n):
            noise[t] = phi * noise[t-1] + innovation[t]
        return noise

    noise_1 = create_ar_noise(base_error_scale * 0.5)
    df_new[f"{target_col_name}_Forecast_1"] = (actual_values + noise_1).round(precision)
    noise_2 = create_ar_noise(base_error_scale * 1.5)
    df_new[f"{target_col_name}_Forecast_2"] = (actual_values + noise_2).round(precision)
    
    cols_to_fix = [f"{target_col_name}_Forecast_1", f"{target_col_name}_Forecast_2"]
    return df_new[[df.columns[0], target_col_name] + cols_to_fix]

## 3. Solar generation data extrapolation from TokWise's dataset

In [ ]:
# Select, for each delivery period, the snapshot closest to delivery start
# to align the latest available forecast with the corresponding actual outcome

Solar = (
    df[["PeriodStartLocal", "fc_mwh_curr", "act_mwh"]]
      .drop_duplicates(subset=["PeriodStartLocal"], keep="first")
      .sort_values("PeriodStartLocal")
      .rename(columns={
          "fc_mwh_curr": "Forecast [MW]",
          "act_mwh":     "Actual [MW]",
      })[["PeriodStartLocal", "Forecast [MW]", "Actual [MW]"]]
)
Solar.head()

In [ ]:
# Extract capacity and convert to float and MW
capacity = float(df["Capacity"].iloc[0])/1000

Solar = Solar.assign(
    Forecast_Load_Factor_1=lambda d: d["Forecast [MW]"].astype(float) / capacity,
    Actual_Load_Factor=lambda d: d["Actual [MW]"].astype(float) / capacity,
)


Solar.head()

## 4. Selecting Data for Model Run

In [ ]:
# The model now runs on the full available horizon from the dataset.

# IMPORTANT: Reload full unfiltered data from raw dataset for forecast generation
# so persistence forecasts can use previous-day values across the whole year.
print("\n=== Reloading full price data from raw dataset ===")
DA_p_full = (
    df[["PeriodStartLocal", "DA"]]
      .drop_duplicates(subset=["PeriodStartLocal"])
      .rename(columns={"DA": "DA_price"})
      .sort_values("PeriodStartLocal")
      .reset_index(drop=True)
)

# Use full dataset date range for analysis/optimization horizon
start = pd.Timestamp(DA_p_full["PeriodStartLocal"].min())
end = pd.Timestamp(DA_p_full["PeriodStartLocal"].max())
print(f"Full-horizon window: {start} -> {end}")

# IMPORTANT: Generate ID forecast from full dataset before filtering
ID_p_full = (
    df[["PeriodStartLocal", "closest_VWAP", "lead_time"]]
      .sort_values(["PeriodStartLocal", "lead_time"])
      .drop_duplicates(subset=["PeriodStartLocal"], keep="first")
      .rename(columns={"closest_VWAP": "ID_price"})[["PeriodStartLocal", "ID_price"]]
      .sort_values("PeriodStartLocal")
      .reset_index(drop=True)
)
ID_p_full["ID_price"] = ID_p_full["ID_price"].ffill().bfill()

# Generate forecasts from FULL unfiltered data
print("=== Generating price forecasts from full dataset ===")
DA_p_forecasts = generate_day_lag_price_forecast(DA_p_full, lag_periods=96)
ID_p_forecasts = generate_synthetic_price_forecasts(ID_p_full)
BCap_Neg_p_forecasts = generate_synthetic_price_forecasts(BCap_Neg_p)
EAM_p_forecasts = generate_synthetic_price_forecasts(EAM_p)
print("✓ Forecasts generated successfully over full horizon.\n")

# Create a complete 15-minute interval index for full horizon
complete_time_index = pd.date_range(start=start, end=end, freq='15min')
print(f"Created complete time index with {len(complete_time_index)} periods")

# Filter and align each dataframe for the selected horizon with complete time index
def filter_and_align(df, start, end, complete_index):
    df_copy = df.copy()
    df_copy["PeriodStartLocal"] = pd.to_datetime(df_copy["PeriodStartLocal"])

    # Handle timezone compatibility: if PeriodStartLocal is timezone-aware,
    # convert start/end/complete_index to match.
    if df_copy["PeriodStartLocal"].dt.tz is not None:
        tz = df_copy["PeriodStartLocal"].dt.tz
        start = pd.Timestamp(start).tz_localize(tz) if start.tz is None else start.tz_convert(tz)
        end = pd.Timestamp(end).tz_localize(tz) if end.tz is None else end.tz_convert(tz)
        if complete_index.tz is None:
            complete_index = complete_index.tz_localize(tz)
        else:
            complete_index = complete_index.tz_convert(tz)

    df_copy = df_copy[(df_copy["PeriodStartLocal"] >= start) & (df_copy["PeriodStartLocal"] <= end)]
    df_copy = df_copy.sort_values("PeriodStartLocal")
    df_copy = df_copy.set_index("PeriodStartLocal")
    df_copy = df_copy.reindex(complete_index)

    # For full-horizon runs, some external series may only cover part of the period.
    # Fill gaps so aligned arrays remain usable by the optimizer.
    df_copy = df_copy.ffill().bfill()

    df_copy = df_copy.reset_index()
    df_copy = df_copy.rename(columns={'index': 'PeriodStartLocal'})

    return df_copy

# Now filter all data to the full analysis horizon
DA_p = filter_and_align(DA_p_full, start, end, complete_time_index)
ID_p = filter_and_align(ID_p_full, start, end, complete_time_index)
Imb_p = filter_and_align(Imb_p, start, end, complete_time_index)
EAM_p = filter_and_align(EAM_p, start, end, complete_time_index)
BCap_Neg_p = filter_and_align(BCap_Neg_p, start, end, complete_time_index)
DA_p_forecasts = filter_and_align(DA_p_forecasts, start, end, complete_time_index)
ID_p_forecasts = filter_and_align(ID_p_forecasts, start, end, complete_time_index)
EAM_p_forecasts = filter_and_align(EAM_p_forecasts, start, end, complete_time_index)
BCap_Neg_p_forecasts = filter_and_align(BCap_Neg_p_forecasts, start, end, complete_time_index)

# Regenerate EEG premium data from full dataset then filter
EEG_p_full = add_market_premium(
    DA_p=DA_p_full,
    periods_per_hour=1,
    reference_value=reference_value_2025,
    market_values=market_values_2025
)
EEG_p = filter_and_align(EEG_p_full, start, end, complete_time_index)

Solar = filter_and_align(Solar, start, end, complete_time_index)

print(f"\nFiltered data verification:")
print(f"  DA_p: {len(DA_p)} periods from {DA_p['PeriodStartLocal'].min()} to {DA_p['PeriodStartLocal'].max()}")
print(f"  ID_p: {len(ID_p)} periods from {DA_p['PeriodStartLocal'].min()} to {ID_p['PeriodStartLocal'].max()}")
print(f"  EAM_p: {len(EAM_p)} periods from {EAM_p['PeriodStartLocal'].min()} to {EAM_p['PeriodStartLocal'].max()}")
print(f"  BCap_Neg_p: {len(BCap_Neg_p)} periods from {BCap_Neg_p['PeriodStartLocal'].min()} to {BCap_Neg_p['PeriodStartLocal'].max()} converted data from 4h blocks to 15 min.")
print(f"  EEG_p: {len(EEG_p)} periods from {EEG_p['PeriodStartLocal'].min()} to {EEG_p['PeriodStartLocal'].max()}")
print(f"  Solar: {len(Solar)} periods from {DA_p['PeriodStartLocal'].min()} to {Solar['PeriodStartLocal'].max()}")

# Verify all aligned dataframes cover the same horizon length
n_expected = len(complete_time_index)
assert len(DA_p) == n_expected, f"Expected {n_expected} periods but got {len(DA_p)}"
assert len(ID_p) == n_expected, f"Expected {n_expected} periods but got {len(ID_p)}"
assert len(EAM_p) == n_expected, f"Expected {n_expected} periods but got {len(EAM_p)}"
assert len(BCap_Neg_p) == n_expected, f"Expected {n_expected} periods but got {len(BCap_Neg_p)}"
assert len(EEG_p) == n_expected, f"Expected {n_expected} periods but got {len(EEG_p)}"
assert len(Solar) == n_expected, f"Expected {n_expected} periods but got {len(Solar)}"
print(f"\n✓ All dataframes have exactly {n_expected} periods.")


### Verify Correct Implementation of Forecast

In [ ]:
# Verify that October 5th forecasts match October 4th actual prices
print("\n=== Verifying Naive Persistence Forecast ===")
print("Checking that October 5th forecasts = October 4th actual prices")

# Get October 4th actual prices from full dataset
oct4_start = pd.Timestamp("2025-10-04 00:00:00").tz_localize("Europe/Berlin")
oct4_end = pd.Timestamp("2025-10-04 23:45:00").tz_localize("Europe/Berlin")
oct4_data = DA_p_full[(DA_p_full['PeriodStartLocal'] >= oct4_start) & (DA_p_full['PeriodStartLocal'] <= oct4_end)]

# Get October 5th forecast from DA_p_forecasts
oct5_forecast = DA_p_forecasts[['PeriodStartLocal', 'DA_price_Forecast_1']]

# Compare
print(f"\nOctober 4th actual prices: {len(oct4_data)} periods")
print(f"  Mean: {oct4_data['DA_price'].mean():.2f} EUR/MWh")
print(f"  Min: {oct4_data['DA_price'].min():.2f}, Max: {oct4_data['DA_price'].max():.2f}")

print(f"\nOctober 5th forecast prices: {len(oct5_forecast)} periods")  
print(f"  Mean: {oct5_forecast['DA_price_Forecast_1'].mean():.2f} EUR/MWh")
print(f"  Min: {oct5_forecast['DA_price_Forecast_1'].min():.2f}, Max: {oct5_forecast['DA_price_Forecast_1'].max():.2f}")

# Check if they match
if len(oct4_data) == len(oct5_forecast):
    are_equal = (oct4_data['DA_price'].values == oct5_forecast['DA_price_Forecast_1'].values).all()
    if are_equal:
        print("\n✓ SUCCESS: October 5th forecasts exactly match October 4th actual prices!")
    else:
        # Check how close they are
        diff = abs(oct4_data['DA_price'].values - oct5_forecast['DA_price_Forecast_1'].values).mean()
        print(f"\n⚠ Mean absolute difference: {diff:.4f} EUR/MWh")
        print("Showing first few values for comparison:")
        comparison = pd.DataFrame({
            'Oct_4_Time': oct4_data['PeriodStartLocal'].iloc[:5].values,
            'Oct_4_Actual': oct4_data['DA_price'].iloc[:5].values,
            'Oct_5_Time': oct5_forecast['PeriodStartLocal'].iloc[:5].values,
            'Oct_5_Forecast': oct5_forecast['DA_price_Forecast_1'].iloc[:5].values
        })
        print(comparison)
else:
    print(f"\n⚠ Length mismatch: Oct 4 has {len(oct4_data)} periods, Oct 5 has {len(oct5_forecast)} periods")

In [ ]:
# More detailed verification - check the data structure
print("\n=== Detailed Data Inspection ===")
print(f"\nDA_p_forecasts shape: {DA_p_forecasts.shape}")
print(f"DA_p_forecasts columns: {DA_p_forecasts.columns.tolist()}")
print(f"\nFirst 5 rows of DA_p_forecasts:")
print(DA_p_forecasts.head())
print(f"\nDate range in DA_p_forecasts:")
print(f"  Min: {DA_p_forecasts['PeriodStartLocal'].min()}")
print(f"  Max: {DA_p_forecasts['PeriodStartLocal'].max()}")

print(f"\nDA_p (after filtering) shape: {DA_p.shape}")
print(f"DA_p columns: {DA_p.columns.tolist()}")
print(f"\nFirst 5 rows of DA_p:")
print(DA_p.head())

# Check if DA_price and DA_price_Forecast_1 are equal for October 5th
if 'DA_price' in DA_p_forecasts.columns:
    matching = (DA_p_forecasts['DA_price'].values == DA_p_forecasts['DA_price_Forecast_1'].values).sum()
    print(f"\n  Rows where actual = forecast: {matching} out of {len(DA_p_forecasts)}")
    
    print(f"\n  Actual prices mean: {DA_p_forecasts['DA_price'].mean():.2f} EUR/MWh")
    print(f"  Forecast prices mean: {DA_p_forecasts['DA_price_Forecast_1'].mean():.2f} EUR/MWh")

In [ ]:
# Check what the shift is actually doing
print("\n=== Investigating the shift operation ===")

# Manually check October 4th and 5th in the full dataset
oct4_start = pd.Timestamp("2025-10-04 00:00:00").tz_localize("Europe/Berlin") 
oct4_end = pd.Timestamp("2025-10-04 23:45:00").tz_localize("Europe/Berlin")
oct5_start = pd.Timestamp("2025-10-05 00:00:00").tz_localize("Europe/Berlin")
oct5_end = pd.Timestamp("2025-10-05 23:45:00").tz_localize("Europe/Berlin")

# Get the data from full dataset
oct4_full = DA_p_full[(DA_p_full['PeriodStartLocal'] >= oct4_start) & (DA_p_full['PeriodStartLocal'] <= oct4_end)].copy()
oct5_full = DA_p_full[(DA_p_full['PeriodStartLocal'] >= oct5_start) & (DA_p_full['PeriodStartLocal'] <= oct5_end)].copy()

oct4_full = oct4_full.sort_values('PeriodStartLocal').reset_index(drop=True)
oct5_full = oct5_full.sort_values('PeriodStartLocal').reset_index(drop=True)

print(f"\nOctober 4th in full dataset:")
print(f"  Rows: {len(oct4_full)}")
print(f"  Mean price: {oct4_full['DA_price'].mean():.2f} EUR/MWh")
print(f"  First 3 prices: {oct4_full['DA_price'].iloc[:3].values}")

print(f"\nOctober 5th in full dataset:")
print(f"  Rows: {len(oct5_full)}")
print(f"  Mean price: {oct5_full['DA_price'].mean():.2f} EUR/MWh")
print(f"  First 3 prices: {oct5_full['DA_price'].iloc[:3].values}")

# Now check what indices they occupy in DA_p_full sorted
DA_p_full_sorted = DA_p_full.sort_values('PeriodStartLocal').reset_index(drop=True)
oct4_indices = DA_p_full_sorted[
    (DA_p_full_sorted['PeriodStartLocal'] >= oct4_start) & 
    (DA_p_full_sorted['PeriodStartLocal'] <= oct4_end)
].index
oct5_indices = DA_p_full_sorted[
    (DA_p_full_sorted['PeriodStartLocal'] >= oct5_start) & 
    (DA_p_full_sorted['PeriodStartLocal'] <= oct5_end)
].index

print(f"\nOctober 4th index range in sorted full dataset: {oct4_indices.min()} to {oct4_indices.max()}")
print(f"October 5th index range in sorted full dataset: {oct5_indices.min()} to {oct5_indices.max()}")
print(f"Difference: {oct5_indices.min() - oct4_indices.min()} rows")

# Check if shift(96) from Oct 5 indices would get Oct 4
expected_oct4_start_idx = oct5_indices.min() - 96
print(f"\nIf we shift Oct 5 back by 96, we'd get index: {expected_oct4_start_idx}")
print(f"October 4th actually starts at index: {oct4_indices.min()}")
if expected_oct4_start_idx == oct4_indices.min():
    print("✓ Shift by 96 should correctly get October 4th!")
else:
    print(f"✗ Shift by 96 gets index {expected_oct4_start_idx}, but Oct 4 is at {oct4_indices.min()}")
    print(f"  Difference: {abs(expected_oct4_start_idx - oct4_indices.min())} rows")

In [ ]:
# Check if DA_p_full is sorted
print("\n=== Checking if DA_p_full is sorted ===")
is_sorted = (DA_p_full['PeriodStartLocal'].diff().dropna() >= pd.Timedelta(0)).all()
print(f"Is DA_p_full sorted chronologically? {is_sorted}")

if not is_sorted:
    print("\n✗ DA_p_full is NOT sorted! This explains why shift(96) doesn't work correctly.")
    print("The .shift() function shifts by row position, so if data is not in chronological order,")
    print("it will get the wrong day's prices.")
    
# Check the first few timestamps to see the ordering
print(f"\nFirst 10 timestamps in DA_p_full:")
print(DA_p_full['PeriodStartLocal'].iloc[:10].values)

In [ ]:
print(DA_p.head())
print(len(DA_p))
print(DA_p.tail())

### Align Input Data

Then all data periods are aligned, including the forecasts.

In [ ]:
# First, align all data to quarter-hourly periods
# Create a common time index based on DA prices (quarter-hourly)
# Use pd.DatetimeIndex to preserve timezone information
time_index = pd.DatetimeIndex(DA_p['PeriodStartLocal'])
n_periods = len(time_index)

# Backward-compatible aliases used below in this notebook section
time_index_96 = time_index
n_periods_96 = n_periods

# Align all price data to the common time index
def align_data_to_time(df, time_col='PeriodStartLocal', align_time=None, value_col=None):
    """Align dataframe to common time index using forward fill"""
    # Ensure align_time is a DatetimeIndex
    if not isinstance(align_time, pd.DatetimeIndex):
        align_time = pd.DatetimeIndex(align_time)
    
    # Convert time_col to datetime and ensure timezone compatibility
    df = df.copy()
    df[time_col] = pd.to_datetime(df[time_col])
    
    # Handle timezone compatibility
    align_tz = align_time.tz
    df_tz = df[time_col].dt.tz
    
    if align_tz is not None and df_tz is None:
        df[time_col] = df[time_col].dt.tz_localize(align_tz)
    elif align_tz is None and df_tz is not None:
        df[time_col] = df[time_col].dt.tz_localize(None)
    elif align_tz is not None and df_tz is not None and align_tz != df_tz:
        df[time_col] = df[time_col].dt.tz_convert(align_tz)
    
    df_aligned = pd.DataFrame(index=align_time)
    df_aligned = df_aligned.join(df.set_index(time_col), how='left')
    if value_col:
        df_aligned = df_aligned[[value_col]].ffill()
    return df_aligned

# -----------------------------
# Align actual price data
# -----------------------------
DA_prices_actual = align_data_to_time(DA_p, value_col='DA_price', align_time=time_index_96)
DA_p_actual = DA_prices_actual.values.flatten()

ID_prices_actual = align_data_to_time(ID_p, value_col='ID_price', align_time=time_index_96)
ID_p_actual = ID_prices_actual.values.flatten()

Imb_prices_actual = align_data_to_time(Imb_p, value_col='Imb_price', align_time=time_index_96)
Imb_p_actual = Imb_prices_actual.values.flatten()

BCap_Neg_prices_actual = align_data_to_time(BCap_Neg_p, value_col='BCap_price', align_time=time_index_96)
BCap_Neg_p_actual = BCap_Neg_prices_actual.values.flatten()

EAM_prices_actual = align_data_to_time(EAM_p, value_col='EAM_price', align_time=time_index_96)
EAM_p_actual = EAM_prices_actual.values.flatten()

# Bug Solver
# Check if index name is already a column
if EEG_p.index.name is not None and EEG_p.index.name not in EEG_p.columns:
    EEG_p = EEG_p.reset_index()

# Optional: check result
print(EEG_p.head())

EEG_p = align_data_to_time(EEG_p, value_col='market premium', align_time=time_index_96)
EEG_p_actual = EEG_p.values.flatten()

# -----------------------------
# Align forecast 1 data
# -----------------------------
DA_prices_f1 = align_data_to_time(DA_p_forecasts, value_col='DA_price_Forecast_1', align_time=time_index_96)
DA_p_f1 = DA_prices_f1.values.flatten()

ID_prices_f1 = align_data_to_time(ID_p_forecasts, value_col='ID_price_Forecast_1', align_time=time_index_96)
ID_p_f1 = ID_prices_f1.values.flatten()

BCap_Neg_f1 = align_data_to_time(BCap_Neg_p_forecasts, value_col='BCap_price_Forecast_1', align_time=time_index_96)
BCap_Neg_p_f1 = BCap_Neg_f1.values.flatten()

EAM_f1 = align_data_to_time(EAM_p_forecasts, value_col='EAM_price_Forecast_1', align_time=time_index_96)
EAM_p_f1 = EAM_f1.values.flatten()

Solar_f1 = align_data_to_time(Solar[['PeriodStartLocal','Forecast_Load_Factor_1']], 
                              value_col='Forecast_Load_Factor_1', align_time=time_index_96)
Solar_f1_arr = Solar_f1.values.flatten()

# -----------------------------
# Align forecast 2 data
# -----------------------------
DA_prices_f2 = align_data_to_time(DA_p_forecasts, value_col='DA_price_Forecast_2', align_time=time_index_96)
DA_p_f2 = DA_prices_f2.values.flatten()

ID_prices_f2 = align_data_to_time(ID_p_forecasts, value_col='ID_price_Forecast_2', align_time=time_index_96)
ID_p_f2 = ID_prices_f2.values.flatten()

BCap_Neg_f2 = align_data_to_time(BCap_Neg_p_forecasts, value_col='BCap_price_Forecast_2', align_time=time_index_96)
BCap_Neg_p_f2 = BCap_Neg_f2.values.flatten()

EAM_f2 = align_data_to_time(EAM_p_forecasts, value_col='EAM_price_Forecast_2', align_time=time_index_96)
EAM_p_f2 = EAM_f2.values.flatten()

#USe forecast 1 of the solar in the same place as forecast 2, assuming they don´t deviate much from 9am to 12 pm.
Solar_f2 = align_data_to_time(Solar[['PeriodStartLocal','Forecast_Load_Factor_1']], 
                              value_col='Forecast_Load_Factor_1', align_time=time_index_96)
Solar_f2_arr = Solar_f2.values.flatten()

# -----------------------------
# Align actual Solar data
# -----------------------------
Solar_actual = align_data_to_time(Solar[['PeriodStartLocal', 'Actual_Load_Factor']], 
                                  value_col='Actual_Load_Factor', align_time=time_index_96)
Solar_actual_arr = Solar_actual.values.flatten()

# -----------------------------
# Calculate generation at each time step
# -----------------------------
Generation_actual = capacity * Solar_actual_arr
Generation_f1 = capacity * Solar_f1_arr
Generation_f2 = capacity * Solar_f2_arr

# -----------------------------
print(f"Time range of quarterly data: {time_index_96[0]} to {time_index_96[-1]}")
print(f"Number of periods: {len(time_index_96)}")
assert len(time_index_96) > 0, "Expected at least one period in selected horizon"

In [ ]:
# ============================================================
# SCALE FACTOR: Multiply generation by 10 to simulate a
# larger asset that can participate in the aFRR market.
# 
# NOTE: The original Solar_04 asset (1.41 MW) cannot meet
# the German aFRR minimum bid of 1 MW after applying the
# 67% reservation margin (max aFRR bid = 0.25 MW < 1 MW).
# Scaling by 10 simulates a 14.1 MW asset where:
#   peak generation ~ 4.3 MW
#   max aFRR bid = 4.3 * 0.67 = 2.88 MW > 1 MW minimum
# This allows the model to demonstrate aFRR participation.
# ============================================================

SCALE_FACTOR = 10

Generation_f1     = Generation_f1     * SCALE_FACTOR
Generation_f2     = Generation_f2     * SCALE_FACTOR
Generation_actual = Generation_actual * SCALE_FACTOR
capacity          = capacity          * SCALE_FACTOR

print(f"Scale factor applied: {SCALE_FACTOR}x")
print(f"New capacity:             {capacity:.2f} MW")
print(f"New Generation_f2 max:    {max(Generation_f2):.2f} MW")
print(f"New max aFRR bid possible: {max(Generation_f2) * 0.67:.2f} MW")
print(f"aFRR minimum bid:          1.00 MW")
print(f"aFRR participation possible: {max(Generation_f2) * 0.67 >= 1.0}")

## 5. Optimization Stages (2-stage model)

In [ ]:
import pulp as pl

# Time set sized to selected analysis horizon
T = list(range(len(time_index_96)))


### Two-Stage MILP Optimization ##

### 5.2 Stage 2: Optimization with Forecast_1 data
Optimize only DA and ID markets, based on forecasts available before DA market closure.


In [ ]:
print("\n=== STAGE 2: Optimization with Forecast_1 data ===")


# Create the optimization model
model_stage2 = pl.LpProblem("Solar_Farm_Stage2", pl.LpMaximize)

# Decision variables (only DA and ID). The lower bound is set on 0, meaning
# that speculative trading by buying/over-selling energy in DA to be sold/bought 
# latter in ID to avoid imblanaces is not allowed.
DA_stage2 = pl.LpVariable.dicts("DA", T, lowBound=0, cat="Continuous")
ID_stage2 = pl.LpVariable.dicts("ID", T, lowBound=0, cat="Continuous")

# Constraints
for t in T:
    # 0. Overall scheduled dispatch
    # The actual generation after curtailment cannot be greater than the forecasted generation in MW. 
    # It is also bound to min=0, based on DA_stage2 and ID_stage 2 definitions
    model_stage2 += DA_stage2[t] + ID_stage2[t] <= Generation_f1[t]

# Objective function: convert all rate terms to 15-min EUR with /4 only here
#In this stage, imbalance pricing is not considered since model is constrained to
# avoid a negative imbalance, and positive imbalance can always be curtailed.
model_stage2 += pl.lpSum(
    DA_stage2[t] / 4 * DA_p_f1[t] +                   # DA revenue (EUR/period)
    ID_stage2[t] / 4 * ID_p_f1[t] +                   # ID revenue (EUR/period)
    (EEG_p_actual[t]* (DA_stage2[t]+ID_stage2[t])/4 ) # EEG premium (EUR/period)
    for t in T
), "Total_Revenue_incl_EEG"


# Solve
model_stage2.solve(pl.PULP_CBC_CMD(msg=0))
print(f"Stage 2 Status: {pl.LpStatus[model_stage2.status]}")
print(f"Stage 2 Objective Value: {pl.value(model_stage2.objective):.2f} EUR")


# Extract results
DA_stage2_results = [DA_stage2[t].varValue for t in T]
ID_stage2_results = [ID_stage2[t].varValue for t in T]
Market_Premium_stage2_results = [
    EEG_p_actual[t] * (DA_stage2[t].varValue + ID_stage2[t].varValue) / 4 
    for t in T]
Generation_real_f1 = [(DA_stage2[t].varValue+ID_stage2[t].varValue) for t in T] # This represents the generation after curtailment.

print("Stage 2 completed. DA is now fixed for Stage 3.")

### 5.3 Stage 3: Optimization with Actual data ##
BCap_Pos, BCap_Neg, and DA are fixed from previous stages; optimize only ID market, run once for each quarter-hour



In [ ]:
print("\n=== STAGE 3: Optimization with fixed DA schedule (ID-only hedging, using actual data) ===")

# Create the optimization model
model_stage3 = pl.LpProblem("Solar_Farm_Stage3", pl.LpMaximize)

# Fix DA schedule from Stage 2 as committed position (earlier DA decisions)
DA_fixed_stage3 = {t: DA_stage2_results[t] for t in T}  # Fixed DA trades

# Imbalance penalty costs (EUR/MWh), set arbitrarily high.
c_pos = 100000.0  # Positive imbalance (surplus) penalty
c_neg = 100000.0  # Negative imbalance (shortage) penalty

# Decision variables: ID market trades (positive=sell generation, negative=buy energy)
# The bounds are the installed capacity, which is the logical limit to compensate for DA deviations or trade the available energy.
ID_stage3 = pl.LpVariable.dicts("ID_trade", T, lowBound=-capacity, upBound=capacity, cat="Continuous")

# Imbalance variables (split positive/negative)
Imb_pos_stage3 = pl.LpVariable.dicts("Imb_pos", T, lowBound=0, cat="Continuous")
Imb_neg_stage3 = pl.LpVariable.dicts("Imb_neg", T, lowBound=0, cat="Continuous")
delta_imb_stage3 = pl.LpVariable.dicts("delta_imb", T, cat="Binary")

# BIG-M for constraints
BIGM = 100000

# Constraints: for each quarter hour
for t in T:
    #The real generation (DA+ ID + imbalances) can not exceed phyisical generation maximum Generation_acutal in MW
    model_stage3 += DA_fixed_stage3[t] + ID_stage3[t] + (Imb_pos_stage3[t] - Imb_neg_stage3[t]) <= Generation_actual[t]

    #The model should have a lower bound for real generation (DA+ ID + imbalances), which cannot be lower than 0 
    model_stage3 += 0 <= DA_fixed_stage3[t] + ID_stage3[t] + (Imb_pos_stage3[t] - Imb_neg_stage3[t])

    # Imbalance exclusivity (can't be both pos and neg)
    model_stage3 += Imb_pos_stage3[t] <= BIGM * delta_imb_stage3[t]
    model_stage3 += Imb_neg_stage3[t] <= BIGM * (1 - delta_imb_stage3[t])
  

# Objective: Maximize revenue - imbalance penalties
model_stage3 += pl.lpSum(
    # DA fixed revenue (actual prices)
    (DA_fixed_stage3[t] * DA_p_actual[t]) / 4 +
    # ID revenue (actual ID prices)
    (ID_stage3[t] * ID_p_actual[t]) / 4 +
    # Imbalance penalties (drive to zero)
    -c_pos * Imb_pos_stage3[t] / 4 -
    c_neg * Imb_neg_stage3[t] / 4 +
    (EEG_p_actual[t]* (DA_fixed_stage3[t]+ID_stage3[t]+ Imb_pos_stage3[t]-Imb_neg_stage3[t])/4 ) # EEG premium (EUR/period)
    for t in T
), "Total_Actual_Revenue"

# Solve
model_stage3.solve(pl.PULP_CBC_CMD(msg=0))
print(f"Stage 3 Status: {pl.LpStatus[model_stage3.status]}")
print(f"Stage 3 Objective Value (EUR): {pl.value(model_stage3.objective):.2f}")

# Extract results
DA_stage3_results = [DA_fixed_stage3[t] for t in T]  # DA remains fixed
ID_stage3_results = [ID_stage3[t].varValue for t in T]
Imb_pos_results = [Imb_pos_stage3[t].varValue for t in T]
Imb_neg_results = [Imb_neg_stage3[t].varValue for t in T]
Market_Premium_stage3_results = [
    EEG_p_actual[t] * (DA_fixed_stage3[t] + ID_stage3[t].varValue + Imb_pos_stage3[t].varValue - Imb_neg_stage3[t].varValue ) / 4 
    for t in T]
Generation_real = [(DA_fixed_stage3[t]+ID_stage3[t].varValue + Imb_pos_stage3[t].varValue - Imb_neg_stage3[t].varValue) for t in T]

print("Stage 3 completed. ID optimized to hedge DA imbalances using actual prices/generation.")

## 6. Results Summary


In [ ]:
results_df = pd.DataFrame({
    'PeriodStartLocal': time_index_96,
    'Generation_Possible_MW': Generation_actual,
    'Generation_Delivered_MW': Generation_real,
    'DA_MW': DA_stage3_results,
    'ID_MW': ID_stage3_results,
    'Imb_pos_MW' : Imb_pos_results,
    'Imb_neg_MW' : Imb_neg_results,
    'Market premium_EUR' : Market_Premium_stage3_results,
    'DA_Price_EUR_MWh': DA_p_actual,
    'ID_Price_EUR_MWh': ID_p_actual,
    'Imb_price' : Imb_p_actual
})

# --- Decompose ID into sell / buy (MW) ---
results_df['ID_Sell_MW'] = results_df['ID_MW'].clip(lower=0)          # ID > 0
results_df['ID_Buy_MW']  = (-results_df['ID_MW'].clip(upper=0))       # -ID where ID < 0

# Calculate revenue components (EUR per 15-min -> /4)
results_df['DA_Revenue_EUR']       = results_df['DA_MW']      / 4 * results_df['DA_Price_EUR_MWh']
results_df['ID_Revenue_EUR']       = results_df['ID_MW']      / 4 * results_df['ID_Price_EUR_MWh']
results_df['ID_Sell_Revenue_EUR']  = results_df['ID_Sell_MW'] / 4 * results_df['ID_Price_EUR_MWh']
results_df['ID_Buy_Cost_EUR']      = results_df['ID_Buy_MW']  / 4 * results_df['ID_Price_EUR_MWh']
#Imbalance settlement is positive if you received money, and negative otherwise
results_df['Imb_Settlement_EUR']   = results_df['Imb_pos_MW']  / 4 * results_df['Imb_price'] - results_df['Imb_neg_MW'] / 4 * results_df['Imb_price']

results_df['Total_Revenue_EUR'] = results_df['DA_Revenue_EUR'] + results_df['ID_Revenue_EUR'] + results_df['Imb_Settlement_EUR'] + results_df['Market premium_EUR']   

# Display summary statistics
print("\n=== OPTIMIZATION RESULTS SUMMARY ===")
print(f"\nTotal Revenue: {results_df['Total_Revenue_EUR'].sum():.2f} EUR")
print(f"\nRevenue Breakdown:")
print(f"  DA Market (net):      {results_df['DA_Revenue_EUR'].sum():.2f} EUR")
print(f"  ID Market (net):      {results_df['ID_Revenue_EUR'].sum():.2f} EUR")
print(f"    ID Sold Revenue:    {results_df['ID_Sell_Revenue_EUR'].sum():.2f} EUR")
print(f"    ID Buy-back Cost:   {-results_df['ID_Buy_Cost_EUR'].sum():.2f} EUR")
print(f"  EEG Market Premium:   {results_df['Market premium_EUR'].sum():.2f} EUR")
print(f"  Imbalance Settlement: {results_df['Imb_Settlement_EUR'].sum():.2f} EUR")

print(f"\nAverage Energy Traded per Period (MW):")
print(f"  DA (net):             {results_df['DA_MW'].mean():.2f} MW")
print(f"  ID (net):             {results_df['ID_MW'].mean():.2f} MW")
print(f"    ID Sold:            {results_df['ID_Sell_MW'].mean():.2f} MW")
print(f"    ID Bought:          {results_df['ID_Buy_MW'].mean():.2f} MW")

print(f"\nTotal Energy Traded (MWh):")
print(f"  Max production possible:  {(results_df['Generation_Possible_MW']/4).sum():.2f} MWh")
print(f"  DA (net):                 {(results_df['DA_MW']/4).sum():.2f} MWh")
print(f"  ID (net):                 {(results_df['ID_MW']/4).sum():.2f} MWh")
print(f"    ID Sold:                {(results_df['ID_Sell_MW']/4).sum():.2f} MWh")
print(f"    ID Bought:              {(results_df['ID_Buy_MW']/4).sum():.2f} MWh")

# Physical curtailment = possible generation - delivered generation
curtailed_energy_mwh = ((results_df['Generation_Possible_MW'] - results_df['Generation_Delivered_MW']) / 4).sum()
total_possible_energy_mwh = (results_df['Generation_Possible_MW'] / 4).sum()
curtailment_rate = (curtailed_energy_mwh / total_possible_energy_mwh) if total_possible_energy_mwh > 0 else np.nan

print(f"  Curtailed Energy:         {curtailed_energy_mwh:.2f} MWh")
print(f"  Curtailment Rate:         {curtailment_rate:.2%}")

# Display first few rows (including explicit ID buy/sell)
print("\n=== First 10 Periods ===")
print(results_df[['PeriodStartLocal',
                  'Generation_Possible_MW',
                  'DA_MW',
                  'ID_MW',
                  'ID_Sell_MW',

                  'ID_Buy_MW',                  'Total_Revenue_EUR']].head(10))

In [ ]:
results_df.head()

In [ ]:
# Curtailment metrics (physical definition)
curtailed_energy_mwh = (
    (results_df['Generation_Possible_MW'] - results_df['Generation_Delivered_MW']) / 4
).clip(lower=0).sum()

total_possible_energy_mwh = (results_df['Generation_Possible_MW'] / 4).sum()
curtailment_rate = (
    curtailed_energy_mwh / total_possible_energy_mwh
    if total_possible_energy_mwh > 0 else np.nan
)

# Temporal concentration: share of curtailment in the top 10% periods
curtailment_per_period_mwh = (
    (results_df['Generation_Possible_MW'] - results_df['Generation_Delivered_MW']) / 4
).clip(lower=0)

n_periods = len(curtailment_per_period_mwh)
top_n = max(1, int(np.ceil(0.10 * n_periods)))
total_curtailment = curtailment_per_period_mwh.sum()
top_10_sum = curtailment_per_period_mwh.nlargest(top_n).sum()
temporal_concentration = (
    top_10_sum / total_curtailment if total_curtailment > 0 else np.nan
)

print(f"Curtailed Energy = {curtailed_energy_mwh:.2f} MWh")
print(f"Curtailment Rate = {curtailment_rate:.2%}")
print(f"Temporal Concentration = {temporal_concentration:.4f}")

# Battery storage relevance assessment based on your threshold table
if np.isnan(curtailment_rate):
    relevance = "N/A"
    interpretation = "No production data available for the selected horizon."
elif curtailment_rate < 0.02:
    relevance = "Low"
    interpretation = "Curtailment is in the <1-2% range, typically indicating sufficient flexibility."
elif curtailment_rate <= 0.04:
    # For 1-4%, temporal concentration helps differentiate whether storage is valuable
    if np.isnan(temporal_concentration):
        relevance = "Moderate"
        interpretation = "Curtailment is in the 1-4% range; storage relevance is moderate."
    elif temporal_concentration >= 0.50:
        relevance = "Moderate to High"
        interpretation = "Curtailment is 1-4% but highly concentrated in few periods, increasing storage value."
    else:
        relevance = "Moderate"
        interpretation = "Curtailment is 1-4% and not strongly concentrated."
elif curtailment_rate > 0.05:
    relevance = "High"
    interpretation = "Curtailment >5% suggests persistent constraints; storage is highly relevant."
else:
    # 4-5% bridge region between table bands
    relevance = "Moderate to High"
    interpretation = "Curtailment is between 4-5%; storage relevance is elevated."

print(f"Battery Storage Relevance = {relevance}")
print(f"Evaluation: {interpretation}")


### 6.0 Benchmark: Naive Strategy (Always Sell Forecast in DA)

In [ ]:
# ===== BENCHMARK MODEL: Always sell all forecasted generation in DA market =====
print("\n" + "="*70)
print("           BENCHMARK: NAIVE DA-ONLY STRATEGY")
print("="*70)
print("\nStrategy: Sell all forecasted generation in DA market (no optimization)")
print("          Accept all imbalances (no ID hedging)")

# Benchmark strategy: DA commitment = forecasted generation
benchmark_DA = Generation_f1  # Sell everything forecasted in DA

# Calculate imbalances (actual - committed)
benchmark_imbalance = Generation_actual - benchmark_DA
benchmark_imb_pos = np.maximum(benchmark_imbalance, 0)  # Surplus
benchmark_imb_neg = np.maximum(-benchmark_imbalance, 0)  # Shortage

# Revenue components
# DA revenue
benchmark_DA_revenue = (benchmark_DA * DA_p_actual / 4).sum()

# EEG premium (only on what's actually delivered)
benchmark_generation_delivered = benchmark_DA - benchmark_imb_neg + benchmark_imb_pos
benchmark_EEG_premium = (EEG_p_actual * benchmark_generation_delivered / 4).sum()

# Imbalance settlement (positive imbalances earn, negative cost)
benchmark_imb_settlement = (benchmark_imb_pos * Imb_p_actual / 4).sum() - (benchmark_imb_neg * Imb_p_actual / 4).sum()

# Total revenue
benchmark_total_revenue = benchmark_DA_revenue + benchmark_EEG_premium + benchmark_imb_settlement

# Print benchmark results
print("\n" + "─"*70)
print("BENCHMARK RESULTS")
print("─"*70)
print(f"\nRevenue Breakdown:")
print(f"  DA Market Revenue:    {benchmark_DA_revenue:>12.2f} EUR")
print(f"  EEG Market Premium:   {benchmark_EEG_premium:>12.2f} EUR")
print(f"  Imbalance Settlement: {benchmark_imb_settlement:>12.2f} EUR")
print(f"  ────────────────────────────────────")
print(f"  Total Revenue:        {benchmark_total_revenue:>12.2f} EUR")

print(f"\nImbalance Statistics:")
print(f"  Total Positive Imb:   {benchmark_imb_pos.sum():>10.2f} MW ({benchmark_imb_pos.sum()/4:.2f} MWh)")
print(f"  Total Negative Imb:   {benchmark_imb_neg.sum():>10.2f} MW ({benchmark_imb_neg.sum()/4:.2f} MWh)")
print(f"  Periods with Imb:     {np.sum((benchmark_imb_pos > 0) | (benchmark_imb_neg > 0))}/96")

# Compare to optimized model
optimized_total_revenue = results_df['Total_Revenue_EUR'].sum()
revenue_improvement = optimized_total_revenue - benchmark_total_revenue
improvement_percentage = (revenue_improvement / benchmark_total_revenue) * 100

print("\n" + "="*70)
print("           COMPARISON: OPTIMIZED vs BENCHMARK")
print("="*70)
print(f"\nOptimized Model Revenue:  {optimized_total_revenue:>12.2f} EUR")
print(f"Benchmark Model Revenue:  {benchmark_total_revenue:>12.2f} EUR")
print(f"────────────────────────────────────────────")
print(f"Revenue Improvement:      {revenue_improvement:>12.2f} EUR")
print(f"Improvement Percentage:   {improvement_percentage:>12.2f} %")

if revenue_improvement > 0:
    print(f"\n✓ The optimized model earns {improvement_percentage:.2f}% MORE than the naive benchmark")
else:
    print(f"\n✗ The benchmark performs better by {-improvement_percentage:.2f}%")

print("="*70)

In [ ]:
# ===== CUMULATIVE REVENUE VISUALIZATION: OPTIMIZED vs BENCHMARK =====
import matplotlib.pyplot as plt
import numpy as np

# Calculate cumulative revenue for OPTIMIZED MODEL (from results_df)
optimized_cumulative_revenue = results_df['Total_Revenue_EUR'].cumsum()

# Calculate cumulative revenue for BENCHMARK MODEL (period by period)
# Need to create period-by-period revenue for benchmark
benchmark_DA_revenue_per_period = benchmark_DA * DA_p_actual / 4
benchmark_EEG_premium_per_period = EEG_p_actual * (benchmark_DA - benchmark_imb_neg + benchmark_imb_pos) / 4
benchmark_imb_settlement_per_period = benchmark_imb_pos * Imb_p_actual / 4 - benchmark_imb_neg * Imb_p_actual / 4
benchmark_total_revenue_per_period = benchmark_DA_revenue_per_period + benchmark_EEG_premium_per_period + benchmark_imb_settlement_per_period

# Calculate cumulative sum for benchmark
benchmark_cumulative_revenue = pd.Series(benchmark_total_revenue_per_period).cumsum()

# Create the plot
fig, ax = plt.subplots(figsize=(14, 7))

# Plot both cumulative revenue curves
ax.plot(results_df['PeriodStartLocal'], optimized_cumulative_revenue, 
        label='Optimized Model (DA + ID hedging)', color='darkgreen', linewidth=2.5, alpha=0.9)
ax.plot(results_df['PeriodStartLocal'], benchmark_cumulative_revenue, 
        label='Benchmark Model (DA only, no ID)', color='darkorange', linewidth=2.5, alpha=0.9, linestyle='--')

# Add shaded area showing the improvement
ax.fill_between(results_df['PeriodStartLocal'], 
                benchmark_cumulative_revenue, 
                optimized_cumulative_revenue,
                where=(optimized_cumulative_revenue >= benchmark_cumulative_revenue),
                interpolate=True,
                alpha=0.2, color='green', label='Revenue Improvement')

# Formatting
ax.set_xlabel('Time (Quarter-Hour Period)', fontsize=12)
ax.set_ylabel('Cumulative Revenue (EUR)', fontsize=12)
ax.set_title('Cumulative Revenue Comparison: Optimized Model vs Benchmark', fontsize=14, fontweight='bold')
ax.legend(loc='upper left', fontsize=11)
ax.grid(True, alpha=0.3)
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')

# Add final revenue values as text annotations
final_optimized = optimized_cumulative_revenue.iloc[-1]
final_benchmark = benchmark_cumulative_revenue.iloc[-1]
improvement = final_optimized - final_benchmark

ax.text(0.02, 0.98, 
        f'Final Optimized: {final_optimized:.2f} EUR\n'
        f'Final Benchmark: {final_benchmark:.2f} EUR\n'
        f'Improvement: +{improvement:.2f} EUR ({improvement_percentage:.2f}%)',
        transform=ax.transAxes, fontsize=10, verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.show()

print(f"\n{'='*70}")
print("CUMULATIVE REVENUE ANALYSIS")
print(f"{'='*70}")
print(f"\nFinal Cumulative Revenue (End of Day):")
print(f"  Optimized Model:  {final_optimized:>12.2f} EUR")
print(f"  Benchmark Model:  {final_benchmark:>12.2f} EUR")
print(f"  ────────────────────────────────────")
print(f"  Improvement:      {improvement:>12.2f} EUR (+{improvement_percentage:.2f}%)")
print(f"{'='*70}\n")

#### 6.0.1 Cumulative Revenue Comparison: Optimized vs Benchmark

### 6.1 Visualization: Generation and Market Allocations Across Stages

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Create figure with 3 subplots
fig, axes = plt.subplots(4, 1, figsize=(16, 16))
fig.suptitle('Optimization Results Across Three Stages', fontsize=16, fontweight='bold')

# X-axis: quarter-hour indices (0-95)
quarter_hours = np.arange(len(DA_stage2_results))

# Plot 1: Generation at each stage (lines)
ax1 = axes[0]
ax1.plot(quarter_hours, Generation_f1, label='Generation Forecast 1 (Stage 2)', linewidth=2, color='#ff7f0e', linestyle=':')
ax1.plot(quarter_hours, Generation_actual, label='Generation Possible (Stage 3)', linewidth=2, color='#2ca02c', linestyle=':')
ax1.plot(quarter_hours, Generation_real_f1, label='Generation After Curtailment 1 (Stage 2)', linewidth=2, color='#ff7f0e', linestyle='-')
ax1.plot(quarter_hours, Generation_real, label='Generation After Curtailment (Stage 3)', linewidth=2, color='#2ca02c', linestyle='-')
ax1.set_xlabel('Quarter-Hour Index', fontsize=12)
ax1.set_ylabel('Generation (MW)', fontsize=12)
ax1.set_title('Solar Generation Forecasts at Each Decision Stage', fontsize=13, fontweight='bold')
ax1.legend(loc='best', fontsize=10)
ax1.grid(True, alpha=0.3)
ax1.set_xlim(0, 95)

# Plot 2: DA volumes at each stage (bars)
ax2 = axes[1]
x = np.arange(len(DA_stage2_results))
width = 0.25  # Width of bars
ax2.bar(x, DA_stage2_results, width, label='DA Stage 2 (Forecast 1)', alpha=0.8, color='#ff7f0e')
ax2.bar(x + width, DA_stage3_results, width, label='DA Stage 3 (Actual)', alpha=0.8, color='#2ca02c')
ax2.set_xlabel('Quarter-Hour Index', fontsize=12)
ax2.set_ylabel('DA Volume (MW)', fontsize=12)
ax2.set_title('Day-Ahead Market Allocations at Each Decision Stage', fontsize=13, fontweight='bold')
ax2.legend(loc='best', fontsize=10)
ax2.grid(True, alpha=0.3, axis='y')
ax2.set_xlim(-0.5, 95.5)

# Plot 3: ID volumes at each stage (bars)
ax3 = axes[2]
ax3.bar(x, ID_stage2_results, width, label='ID Stage 2 (Forecast 1)', alpha=0.8, color='#ff7f0e')
ax3.bar(x + width, ID_stage3_results, width, label='ID Stage 3 (Actual)', alpha=0.8, color='#2ca02c')
ax3.set_xlabel('Quarter-Hour Index', fontsize=12)
ax3.set_ylabel('ID Volume (MW)', fontsize=12)
ax3.set_title('Intraday Market Allocations at Each Decision Stage', fontsize=13, fontweight='bold')
ax3.legend(loc='best', fontsize=10)
ax3.grid(True, alpha=0.3, axis='y')
ax3.set_xlim(-0.5, 95.5)

# Plot 4: Stage 3 – Generation + Market Stack (Single Axis)
ax4 = axes[3]
x = np.arange(len(DA_stage3_results))
width = 0.6

# Convert arrays
DA = np.array(DA_stage3_results)
ID = np.array(ID_stage3_results)
IMB_POS = np.array(Imb_pos_results)
IMB_NEG = np.array(Imb_neg_results)

# Split ID into positive and negative
ID_pos = np.where(ID > 0, ID, 0)
ID_neg = np.where(ID < 0, ID, 0)

# ----- Positive stack -----
market_positive = {
    "DA (Stage 3)": DA,
    "ID Positive": ID_pos,
    "Imbalance Positive": IMB_POS,
}

bottom_pos = np.zeros(len(x))
for label, values in market_positive.items():
    ax4.bar(x, values, width, bottom=bottom_pos, label=label, alpha=0.8)
    bottom_pos += values

# ----- Negative stack -----
market_negative = {
    "ID Negative": ID_neg,
    "Imbalance Negative": -IMB_NEG,  # plot downward
}

bottom_neg = np.zeros(len(x))
for label, values in market_negative.items():
    ax4.bar(x, values, width, bottom=bottom_neg, label=label, alpha=0.8)
    bottom_neg += values

# ----- Generation lines -----
ax4.plot(quarter_hours, Generation_actual, lw=2, ls=':', label='Generation Possible (Stage 3)')
ax4.plot(quarter_hours, Generation_real, lw=2, ls='-', label='Generation After Curtailment (Stage 3)')

# ----- Axis formatting -----
ax4.set(xlabel='Quarter-Hour Index', ylabel='MW', xlim=(0,95),
        title='Stage 3: Generation and Market Allocation')
ax4.axhline(0, linewidth=1)
ax4.grid(True, alpha=0.3)

# Set y-limits to include negative bars
all_values = np.concatenate([bottom_pos, bottom_neg, Generation_actual, Generation_real])
ax4.set_ylim(all_values.min()*1.1, all_values.max()*1.1)

# ----- Legend -----
h1, l1 = ax4.get_legend_handles_labels()
ax4.legend(h1, l1, loc='best', fontsize=9)




plt.tight_layout()
plt.show()

# Print summary statistics
print("\n=== Summary Statistics Across Stages ===")
print(f"\nGeneration (MW):")
print(f"  Stage 2 (Forecast 1):  Min={Generation_f1.min():.2f}, Max={Generation_f1.max():.2f}, Mean={Generation_f1.mean():.2f}")
print(f"  Stage 3 (Actual):     Min={Generation_actual.min():.2f}, Max={Generation_actual.max():.2f}, Mean={Generation_actual.mean():.2f}")

print(f"\nDA Allocations (MW):")
#print(f"  Stage 1 Total: {sum(DA_stage1_results):.2f} MW ({sum(DA_stage1_results)/4:.2f} MWh)")
print(f"  Stage 2 Total: {sum(DA_stage2_results):.2f} MW ({sum(DA_stage2_results)/4:.2f} MWh)")
print(f"  Stage 3 Total: {sum(DA_stage3_results):.2f} MW ({sum(DA_stage3_results)/4:.2f} MWh)")

print(f"\nID Allocations (MW):")
#print(f"  Stage 1 Total: {sum(ID_stage1_results):.2f} MW ({sum(ID_stage1_results)/4:.2f} MWh)")
print(f"  Stage 2 Total: {sum(ID_stage2_results):.2f} MW ({sum(ID_stage2_results)/4:.2f} MWh)")
print(f"  Stage 3 Total: {sum(ID_stage3_results):.2f} MW ({sum(ID_stage3_results)/4:.2f} MWh)")


In [ ]:
# Create figure with 4 subplots
fig, axes = plt.subplots(4, 1, figsize=(16, 18))
fig.suptitle('DA and IDA Market Analysis', fontsize=16, fontweight='bold')

# ===== Graph 1: Revenue and Traded Volume with Double Y-axis =====
ax1 = axes[0]
ax2 = ax1.twinx()

# Plot revenue on left y-axis
line1 = ax1.plot(results_df['PeriodStartLocal'], results_df['DA_Revenue_EUR'], 
                 label='DA Revenue', color='blue', linewidth=2, alpha=0.7)
line2 = ax1.plot(results_df['PeriodStartLocal'], results_df['ID_Revenue_EUR'], 
                 label='IDA Revenue', color='red', linewidth=2, alpha=0.7)
line5 = ax1.plot(results_df['PeriodStartLocal'], results_df['Market premium_EUR'], 
                 label='EEG Premium', color='green', linewidth=2, alpha=0.7)

# Plot traded volume on right y-axis
line3 = ax2.plot(results_df['PeriodStartLocal'], results_df['DA_MW'], 
                 label='DA Volume', color='cyan', linewidth=2, linestyle='--', alpha=0.7)
line4 = ax2.plot(results_df['PeriodStartLocal'], results_df['ID_MW'], 
                 label='IDA Volume', color='orange', linewidth=2, linestyle='--', alpha=0.7)

# Configure left y-axis (Revenue)
ax1.set_xlabel('PeriodStartLocal', fontsize=12)
ax1.set_ylabel('Revenue (EUR)', fontsize=12, color='black')
ax1.tick_params(axis='y', labelcolor='black')
ax1.grid(True, alpha=0.3)

# Configure right y-axis (Volume)
ax2.set_ylabel('Traded Volume (MW)', fontsize=12, color='black')
ax2.tick_params(axis='y', labelcolor='black')

# Combine legends
lines = line1 + line2 + line5 + line3 + line4
labels = [l.get_label() for l in lines]
ax1.legend(lines, labels, loc='upper left', fontsize=10)

ax1.set_title('Revenue and Traded Volume per Timestamp', fontsize=14)
plt.setp(ax1.xaxis.get_majorticklabels(), rotation=45, ha='right')

# ===== Graph 2: Revenue per Traded Volume =====
ax3 = axes[1]

# Calculate revenue per traded volume (EUR/MW for quarter-hour period)
# Avoid division by zero
da_rev_per_vol = np.where(results_df['DA_MW'] != 0, 
                          results_df['DA_Revenue_EUR'] / results_df['DA_MW'], 
                          0)
ida_rev_per_vol = np.where(results_df['ID_MW'] != 0, 
                           results_df['ID_Revenue_EUR'] / results_df['ID_MW'], 
                           0)

# Plot revenue per volume
ax3.plot(results_df['PeriodStartLocal'], da_rev_per_vol, 
         label='DA Revenue/Volume', color='blue', linewidth=2, alpha=0.7)
ax3.plot(results_df['PeriodStartLocal'], ida_rev_per_vol, 
         label='IDA Revenue/Volume', color='red', linewidth=2, alpha=0.7)

ax3.set_xlabel('PeriodStartLocal', fontsize=12)
ax3.set_ylabel('Revenue per Volume (EUR/MW)', fontsize=12)
ax3.set_title('Revenue per Traded Volume', fontsize=14)
ax3.legend(loc='upper left', fontsize=10)
ax3.grid(True, alpha=0.3)
plt.setp(ax3.xaxis.get_majorticklabels(), rotation=45, ha='right')

# ===== Graph 3: Cumulative Revenue =====
ax4 = axes[2]

# Calculate cumulative revenue
cumulative_da_revenue = results_df['DA_Revenue_EUR'].cumsum()
cumulative_ida_revenue = results_df['ID_Revenue_EUR'].cumsum()
cumulative_eeg_premium = results_df['Market premium_EUR'].cumsum()
cumulative_imb_settlement = results_df['Imb_Settlement_EUR'].cumsum()

# Plot cumulative revenue
ax4.plot(results_df['PeriodStartLocal'], cumulative_da_revenue, 
         label='DA Cumulative Revenue', color='blue', linewidth=2, alpha=0.7)
ax4.plot(results_df['PeriodStartLocal'], cumulative_ida_revenue, 
         label='IDA Cumulative Revenue', color='red', linewidth=2, alpha=0.7)
ax4.plot(results_df['PeriodStartLocal'], cumulative_eeg_premium, 
         label='EEG Premium Cumulative', color='green', linewidth=2, alpha=0.7)
ax4.plot(results_df['PeriodStartLocal'], cumulative_da_revenue + cumulative_ida_revenue + cumulative_eeg_premium + cumulative_imb_settlement, 
         label='Total Cumulative Revenue', color='purple', linewidth=2.5, alpha=0.8, linestyle=':')

ax4.set_xlabel('PeriodStartLocal', fontsize=12)
ax4.set_ylabel('Cumulative Revenue (EUR)', fontsize=12)
ax4.set_title('Cumulative Revenue Over Time', fontsize=14)
ax4.legend(loc='upper left', fontsize=10)
ax4.grid(True, alpha=0.3)
plt.setp(ax4.xaxis.get_majorticklabels(), rotation=45, ha='right')


ax5 = axes[3]
cumulative_da_volume = (results_df['DA_MW']/4).cumsum()
# Calculate cumulative traded volume (in MWh)
cumulative_da_volume = (results_df['DA_MW']/4).cumsum()
cumulative_ida_volume = (results_df['ID_MW']/4).cumsum()

# Plot cumulative volume
ax5.plot(results_df['PeriodStartLocal'], cumulative_da_volume, 
         label='DA Cumulative Volume', color='blue', linewidth=2, alpha=0.7)
ax5.plot(results_df['PeriodStartLocal'], cumulative_ida_volume, 
         label='IDA Cumulative Volume', color='red', linewidth=2, alpha=0.7)
ax5.plot(results_df['PeriodStartLocal'], cumulative_da_volume + cumulative_ida_volume, 
         label='Total Cumulative Volume', color='green', linewidth=2, alpha=0.7, linestyle=':')

ax5.set_xlabel('PeriodStartLocal', fontsize=12)
ax5.set_ylabel('Cumulative Volume (MWh)', fontsize=12)
ax5.set_title('Cumulative Traded Volume Over Time', fontsize=14)
ax5.legend(loc='upper left', fontsize=10)
ax5.grid(True, alpha=0.3)
plt.setp(ax5.xaxis.get_majorticklabels(), rotation=45, ha='right')

plt.tight_layout()
plt.show()

# Print summary statistics
print("\n=== Revenue and Volume Statistics ===")
print(f"\nDay-Ahead Market:")
print(f"  Total Revenue:        {results_df['DA_Revenue_EUR'].sum():.2f} EUR")
print(f"  Total Volume:         {(results_df['DA_MW']/4).sum():.2f} MWh")
print(f"  Avg Revenue/Volume:   {(results_df['DA_Revenue_EUR'].sum() / (results_df['DA_MW']/4).sum()):.2f} EUR/MWh")

print(f"\nIntraday Market:")
print(f"  Total Revenue:        {results_df['ID_Revenue_EUR'].sum():.2f} EUR")
print(f"  Total Volume:         {(results_df['ID_MW']/4).sum():.2f} MWh")
if (results_df['ID_MW']/4).sum() != 0:
    print(f"  Avg Revenue/Volume:   {(results_df['ID_Revenue_EUR'].sum() / (results_df['ID_MW']/4).sum()):.2f} EUR/MWh")
else:
    print(f"  Avg Revenue/Volume:   N/A (no volume traded)")

print(f"\nEEG Market Premium:")
print(f"  Total Premium:        {results_df['Market premium_EUR'].sum():.2f} EUR")
print(f"  Avg Premium/Period:   {results_df['Market premium_EUR'].mean():.2f} EUR")

print(f"\nImbalance Settlement:")
print(f"  Total Settlement:     {results_df['Imb_Settlement_EUR'].sum():.2f} EUR")
print(f"  Avg Settlement/Period: {results_df['Imb_Settlement_EUR'].mean():.2f} EUR")

print(f"\nOverall Total Revenue: {results_df['Total_Revenue_EUR'].sum():.2f} EUR")

In [ ]:
# Create a dataframe with revenue from IDA and DA per timestamp
revenue_per_timestep = pd.DataFrame({
    'Timestamp': results_df['PeriodStartLocal'],
    'DA_Revenue_EUR': results_df['DA_Revenue_EUR'],
    'DA_Volume_MW': results_df['DA_MW'],
    'IDA_Revenue_EUR': results_df['ID_Revenue_EUR'],
    'IDA_Volume_MW': results_df['ID_MW'],
    'Total_Revenue_EUR': results_df['DA_Revenue_EUR'] + results_df['ID_Revenue_EUR']
})

print("=== Revenue per Timestamp DataFrame ===")
print(f"\nDataFrame shape: {revenue_per_timestep.shape}")
print(f"\nFirst 10 rows:")
print(revenue_per_timestep.head(10))

# Show the 5 most profitable hours for DA
print("\n=== 5 Most Profitable Hours for Day-Ahead Market ===")
top_da = revenue_per_timestep.nlargest(5, 'DA_Revenue_EUR')[['Timestamp', 'DA_Revenue_EUR', 'DA_Volume_MW']]
print(top_da.to_string(index=False))

# Show the 5 most profitable hours for IDA
print("\n=== 5 Most Profitable Hours for Intraday Market ===")
top_ida = revenue_per_timestep.nlargest(5, 'IDA_Revenue_EUR')[['Timestamp', 'IDA_Revenue_EUR', 'IDA_Volume_MW']]
print(top_ida.to_string(index=False))

### 6.2 Imbalance Analysis and Cost

In [ ]:
# ===== IMBALANCE ANALYSIS =====
print("="*60)
print("           IMBALANCE ANALYSIS - STAGE 3 RESULTS")
print("="*60)

# Imbalance data from results_df
imbalance_data = results_df[['PeriodStartLocal', 'Imb_pos_MW', 'Imb_neg_MW', 'Imb_price', 'Imb_Settlement_EUR']].copy()

# Calculate total imbalances
total_imb_pos_MW = results_df['Imb_pos_MW'].sum()
total_imb_neg_MW = results_df['Imb_neg_MW'].sum()

# Calculate total imbalance in MWh (dividing by 4 for 15-min periods)
total_imb_pos_MWh = total_imb_pos_MW / 4
total_imb_neg_MWh = total_imb_neg_MW / 4

# Calculate total imbalance settlement (cost/revenue)
# Positive imbalance = surplus (received money)
# Negative imbalance = shortage (paid money)
total_imb_settlement = results_df['Imb_Settlement_EUR'].sum()

# Count periods with imbalances
periods_with_pos_imb = (results_df['Imb_pos_MW'] > 0.001).sum()
periods_with_neg_imb = (results_df['Imb_neg_MW'] > 0.001).sum()
periods_with_any_imb = periods_with_pos_imb + periods_with_neg_imb

print(f"\n{'─'*60}")
print(f"SUMMARY STATISTICS")
print(f"{'─'*60}")
print(f"\nTotal Periods Analyzed:           {len(results_df)}")
print(f"Periods with Positive Imbalance:  {periods_with_pos_imb} ({periods_with_pos_imb/len(results_df)*100:.1f}%)")
print(f"Periods with Negative Imbalance:  {periods_with_neg_imb} ({periods_with_neg_imb/len(results_df)*100:.1f}%)")
print(f"Periods with No Imbalance:        {len(results_df) - periods_with_any_imb} ({(len(results_df) - periods_with_any_imb)/len(results_df)*100:.1f}%)")

print(f"\n{'─'*60}")
print(f"IMBALANCE VOLUMES")
print(f"{'─'*60}")
print(f"\nPositive Imbalance (Surplus):")
print(f"  Total:  {total_imb_pos_MW:.2f} MW  ({total_imb_pos_MWh:.2f} MWh)")
print(f"  Max:    {results_df['Imb_pos_MW'].max():.2f} MW")
print(f"  Avg:    {results_df['Imb_pos_MW'].mean():.2f} MW")

print(f"\nNegative Imbalance (Shortage):")
print(f"  Total:  {total_imb_neg_MW:.2f} MW  ({total_imb_neg_MWh:.2f} MWh)")
print(f"  Max:    {results_df['Imb_neg_MW'].max():.2f} MW")
print(f"  Avg:    {results_df['Imb_neg_MW'].mean():.2f} MW")

print(f"\n{'─'*60}")
print(f"IMBALANCE SETTLEMENT (FINANCIAL IMPACT)")
print(f"{'─'*60}")
print(f"\nTotal Imbalance Settlement:       {total_imb_settlement:.2f} EUR")
print(f"  (Positive = Revenue, Negative = Cost)")

if total_imb_settlement >= 0:
    print(f"\n✓ Net Revenue from Imbalance: {total_imb_settlement:.2f} EUR")
else:
    print(f"\n✗ Net Cost from Imbalance: {abs(total_imb_settlement):.2f} EUR")

print(f"\nAverage Settlement per Period:    {results_df['Imb_Settlement_EUR'].mean():.2f} EUR")
print(f"Average Imbalance Price:          {results_df['Imb_price'].mean():.2f} EUR/MWh")

print(f"\n{'─'*60}")
print(f"PERIODS WITH HIGHEST IMBALANCES")
print(f"{'─'*60}")

# Show top 5 periods with highest positive imbalance
print("\nTop 5 Positive Imbalances (Surplus):")
top_pos_imb = imbalance_data.nlargest(5, 'Imb_pos_MW')[['PeriodStartLocal', 'Imb_pos_MW', 'Imb_price', 'Imb_Settlement_EUR']]
for idx, row in top_pos_imb.iterrows():
    if row['Imb_pos_MW'] > 0.001:
        print(f"  {row['PeriodStartLocal']}: {row['Imb_pos_MW']:6.2f} MW at {row['Imb_price']:7.2f} EUR/MWh = {row['Imb_Settlement_EUR']:8.2f} EUR")

# Show top 5 periods with highest negative imbalance
print("\nTop 5 Negative Imbalances (Shortage):")
top_neg_imb = imbalance_data.nlargest(5, 'Imb_neg_MW')[['PeriodStartLocal', 'Imb_neg_MW', 'Imb_price', 'Imb_Settlement_EUR']]
for idx, row in top_neg_imb.iterrows():
    if row['Imb_neg_MW'] > 0.001:
        print(f"  {row['PeriodStartLocal']}: {row['Imb_neg_MW']:6.2f} MW at {row['Imb_price']:7.2f} EUR/MWh = {row['Imb_Settlement_EUR']:8.2f} EUR")

print(f"\n{'='*60}")

In [ ]:
# Demonstrate the imbalance avoidance mechanism
print("="*70)
print("         IMBALANCE AVOIDANCE MECHANISM ANALYSIS")
print("="*70)

# Calculate the mismatch that ID market is hedging
results_df['DA_Generation_Mismatch'] = results_df['DA_MW'] - results_df['Generation_Delivered_MW']

# Show statistics
print("\n" + "─"*70)
print("MISMATCH BETWEEN DA COMMITMENTS AND ACTUAL GENERATION")
print("─"*70)
print(f"\nWithout ID market, there would be mismatches!")
print(f"Average DA vs Actual mismatch: {results_df['DA_Generation_Mismatch'].mean():.2f} MW")
print(f"Max positive mismatch: {results_df['DA_Generation_Mismatch'].max():.2f} MW (over-commitment)")
print(f"Max negative mismatch: {results_df['DA_Generation_Mismatch'].min():.2f} MW (under-commitment)")

print("\n" + "─"*70)
print("ID MARKET IS USED TO HEDGE THESE MISMATCHES")
print("─"*70)

# Calculate ID market hedging
id_buys = results_df[results_df['ID_MW'] < -0.001].copy()
id_sells = results_df[results_df['ID_MW'] > 0.001].copy()

print(f"\nPeriods where ID market BUYS power: {len(id_buys)}")
if len(id_buys) > 0:
    print(f"  Total bought: {(-id_buys['ID_MW'].sum()/4):.2f} MWh")
    print(f"  Cost: {id_buys['ID_Revenue_EUR'].sum():.2f} EUR")
    print(f"  Avg price paid: {(id_buys['ID_Revenue_EUR'].sum() / (-id_buys['ID_MW'].sum()/4)):.2f} EUR/MWh")

print(f"\nPeriods where ID market SELLS power: {len(id_sells)}")
if len(id_sells) > 0:
    print(f"  Total sold: {(id_sells['ID_MW'].sum()/4):.2f} MWh")
    print(f"  Revenue: {id_sells['ID_Revenue_EUR'].sum():.2f} EUR")
    print(f"  Avg price received: {(id_sells['ID_Revenue_EUR'].sum() / (id_sells['ID_MW'].sum()/4)):.2f} EUR/MWh")

print("\n" + "─"*70)
print("COST COMPARISON: ID HEDGING vs IMBALANCE PENALTIES")
print("─"*70)

# What would the cost be if we had imbalances instead of hedging?
theoretical_imbalance_volume = abs(results_df['DA_Generation_Mismatch']).sum() / 4  # MWh
actual_id_cost = results_df['ID_Revenue_EUR'].sum()  # Negative = cost, positive = revenue

# Calculate what it would cost with imbalance penalties
imbalance_penalty = 100000.0  # EUR/MWh
theoretical_imbalance_cost = theoretical_imbalance_volume * imbalance_penalty

print(f"\nActual ID market net revenue: {actual_id_cost:.2f} EUR")
print(f"\nIf these were imbalances instead:")
print(f"  Volume to balance: {theoretical_imbalance_volume:.2f} MWh")
print(f"  Cost at 100,000 EUR/MWh: {theoretical_imbalance_cost:.2f} EUR")
print(f"\n💡 Savings by using ID market: {theoretical_imbalance_cost + actual_id_cost:.2f} EUR")
print(f"   (This is why the optimizer chooses ID market over imbalances!)")

print("\n" + "─"*70)
print("BALANCE VERIFICATION")
print("─"*70)

# Verify the balance equation for a few sample periods
print("\nBalance equation: DA + ID + (Imb_pos - Imb_neg) = Generation_Delivered")
print("\nSample periods:")
for i in [0, 20, 40, 60, 80]:
    row = results_df.iloc[i]
    lhs = row['DA_MW'] + row['ID_MW'] + (row['Imb_pos_MW'] - row['Imb_neg_MW'])
    rhs = row['Generation_Delivered_MW']
    match = "✓" if abs(lhs - rhs) < 0.01 else "✗"
    print(f"  Period {i}: {row['DA_MW']:.2f} + {row['ID_MW']:.2f} + 0.00 = {row['Generation_Delivered_MW']:.2f} {match}")

print("\n" + "="*70)

In [ ]:
# ===== GENERATION FORECAST ERROR ANALYSIS =====
print("="*75)
print("        GENERATION FORECAST vs ACTUAL - ERROR ANALYSIS")
print("="*75)

# Calculate forecast errors
forecast_error_MW = Generation_f1 - Generation_actual
forecast_error_pct = ((Generation_f1 - Generation_actual) / (Generation_f1 + 0.001)) * 100

print("\n" + "─"*75)
print("SUMMARY STATISTICS")
print("─"*75)

print(f"\nGeneration Forecast 1 (Stage 2):")
print(f"  Minimum:  {Generation_f1.min():.2f} MW")
print(f"  Maximum:  {Generation_f1.max():.2f} MW")
print(f"  Average:  {Generation_f1.mean():.2f} MW")
print(f"  Total:    {(Generation_f1.sum()/4):.2f} MWh")

print(f"\nGeneration Actual (Stage 3):")
print(f"  Minimum:  {Generation_actual.min():.2f} MW")
print(f"  Maximum:  {Generation_actual.max():.2f} MW")
print(f"  Average:  {Generation_actual.mean():.2f} MW")
print(f"  Total:    {(Generation_actual.sum()/4):.2f} MWh")

print(f"\nForecast Error (Forecast - Actual):")
print(f"  Average:        {forecast_error_MW.mean():.2f} MW")
print(f"  Std Deviation:  {forecast_error_MW.std():.2f} MW")
print(f"  Max Over-forecast:  {forecast_error_MW.max():.2f} MW")
print(f"  Max Under-forecast: {forecast_error_MW.min():.2f} MW")

# Calculate metrics
mae = np.abs(forecast_error_MW).mean()
mape = np.abs(forecast_error_pct).mean()
rmse = np.sqrt(np.mean(forecast_error_MW**2))

print(f"\n" + "─"*75)
print("FORECAST ERROR METRICS")
print("─"*75)
print(f"\nMean Absolute Error (MAE):       {mae:.2f} MW")
print(f"Root Mean Square Error (RMSE):   {rmse:.2f} MW")
print(f"Mean Absolute Percentage Error:  {mape:.1f}%")

# Over-forecasting analysis
over_forecast = forecast_error_MW > 0.1
under_forecast = forecast_error_MW < -0.1
accurate = (forecast_error_MW >= -0.1) & (forecast_error_MW <= 0.1)

print(f"\n" + "─"*75)
print("FORECAST BIAS ANALYSIS")
print("─"*75)
print(f"\nPeriods with OVER-forecast:   {over_forecast.sum()} ({over_forecast.sum()/len(forecast_error_MW)*100:.1f}%)")
print(f"  Total excess predicted:     {forecast_error_MW[over_forecast].sum():.2f} MW")
print(f"Periods with UNDER-forecast:  {under_forecast.sum()} ({under_forecast.sum()/len(forecast_error_MW)*100:.1f}%)")
print(f"  Total deficit predicted:    {abs(forecast_error_MW[under_forecast].sum()):.2f} MW")
print(f"Periods with accurate forecast: {accurate.sum()} ({accurate.sum()/len(forecast_error_MW)*100:.1f}%)")

print(f"\n💡 Forecast BIAS: {forecast_error_MW.mean():.2f} MW")
if forecast_error_MW.mean() > 1:
    print("   → Strong POSITIVE BIAS: Forecast consistently OVER-estimates actual generation")
elif forecast_error_MW.mean() < -1:
    print("   → Strong NEGATIVE BIAS: Forecast consistently UNDER-estimates actual generation")
else:
    print("   → Relatively unbiased forecast")



# Calculate financial impact of forecast error
revenue_if_forecast_correct = (Generation_f1.sum()/4) * DA_p_actual.mean()
actual_revenue = (Generation_actual.sum()/4) * DA_p_actual.mean()
lost_opportunity = revenue_if_forecast_correct - actual_revenue

print(f"\nIf forecast was accurate:")
print(f"  Could generate: {Generation_f1.sum()/4:.2f} MWh")
print(f"  Estimated revenue: {revenue_if_forecast_correct:.2f} EUR")
print(f"\nActual generation:")
print(f"  Generated: {Generation_actual.sum()/4:.2f} MWh")
print(f"  Estimated revenue: {actual_revenue:.2f} EUR")
print(f"\nLost opportunity due to lower-than-forecast generation:")
print(f"  {(Generation_f1.sum() - Generation_actual.sum())/4:.2f} MWh")
print(f"  ~{lost_opportunity:.2f} EUR (at avg DA price)")

print("\n" + "="*75)

In [ ]:
# ===== VISUALIZATION: FORECAST vs ACTUAL =====
import matplotlib.pyplot as plt

fig, axes = plt.subplots(3, 1, figsize=(16, 12))
fig.suptitle('Generation Forecast vs Actual - Detailed Analysis', fontsize=16, fontweight='bold')

# Create time axis dynamically from data length
n_periods_plot = len(Generation_f1)
time_periods = np.arange(n_periods_plot)
time_labels = [f"{i//4:02d}:{(i%4)*15:02d}" for i in range(n_periods_plot)]

# Plot 1: Forecast vs Actual Generation
ax1 = axes[0]
ax1.plot(time_periods, Generation_f1, label='Forecast (Stage 2)', 
         color='orange', linewidth=2.5, marker='o', markersize=3, alpha=0.7)
ax1.plot(time_periods, Generation_actual, label='Actual (Stage 3)', 
         color='green', linewidth=2.5, marker='s', markersize=3, alpha=0.7)
ax1.fill_between(time_periods, Generation_f1, Generation_actual, 
                  where=(Generation_f1 >= Generation_actual), 
                  color='red', alpha=0.2, label='Over-forecast')
ax1.fill_between(time_periods, Generation_f1, Generation_actual, 
                  where=(Generation_f1 < Generation_actual), 
                  color='blue', alpha=0.2, label='Under-forecast')
ax1.set_xlabel('Quarter-Hour Period', fontsize=12)
ax1.set_ylabel('Generation (MW)', fontsize=12)
ax1.set_title('Generation: Forecast vs Actual', fontsize=14, fontweight='bold')
ax1.legend(loc='best', fontsize=10)
ax1.grid(True, alpha=0.3)
ax1.set_xlim(0, n_periods_plot - 1)

# Plot 2: Forecast Error (MW)
ax2 = axes[1]
forecast_error_MW = Generation_f1 - Generation_actual
colors = ['red' if e > 0 else 'blue' for e in forecast_error_MW]
ax2.bar(time_periods, forecast_error_MW, color=colors, alpha=0.6, width=0.8)
ax2.axhline(y=0, color='black', linestyle='-', linewidth=1)
ax2.axhline(y=forecast_error_MW.mean(), color='red', linestyle='--', 
            linewidth=2, label=f'Mean Error: {forecast_error_MW.mean():.2f} MW')
ax2.set_xlabel('Quarter-Hour Period', fontsize=12)
ax2.set_ylabel('Forecast Error (MW)', fontsize=12)
ax2.set_title('Forecast Error (Positive = Over-forecast, Negative = Under-forecast)', 
              fontsize=14, fontweight='bold')
ax2.legend(loc='best', fontsize=10)
ax2.grid(True, alpha=0.3, axis='y')
ax2.set_xlim(0, n_periods_plot - 1)

# Plot 3: Forecast Accuracy (%)
ax3 = axes[2]
forecast_accuracy = (1 - np.abs(forecast_error_MW) / (Generation_f1 + 0.001)) * 100
forecast_accuracy = np.clip(forecast_accuracy, 0, 100)  # Clip to 0-100%
ax3.plot(time_periods, forecast_accuracy, color='purple', linewidth=2, marker='o', markersize=3)
ax3.axhline(y=forecast_accuracy.mean(), color='red', linestyle='--', 
            linewidth=2, label=f'Mean Accuracy: {forecast_accuracy.mean():.1f}%')
ax3.axhline(y=80, color='green', linestyle=':', linewidth=1, alpha=0.5, label='80% Threshold')
ax3.fill_between(time_periods, 0, forecast_accuracy, 
                  where=(forecast_accuracy >= 80), color='green', alpha=0.2)
ax3.fill_between(time_periods, 0, forecast_accuracy, 
                  where=(forecast_accuracy < 80), color='red', alpha=0.2)
ax3.set_xlabel('Quarter-Hour Period', fontsize=12)
ax3.set_ylabel('Forecast Accuracy (%)', fontsize=12)
ax3.set_title('Forecast Accuracy (100% = Perfect Forecast)', fontsize=14, fontweight='bold')
ax3.legend(loc='best', fontsize=10)
ax3.grid(True, alpha=0.3)
ax3.set_xlim(0, n_periods_plot - 1)
ax3.set_ylim(0, 105)

plt.tight_layout()
plt.show()

### 6.5 DA and ID Market Prices

In [ ]:
# Plot DA and ID market prices (actual and forecast)
fig, axes = plt.subplots(2, 1, figsize=(14, 10))
time_periods = np.arange(len(DA_p_actual))

# Subplot 1: Day-Ahead Market Prices
ax1 = axes[0]
ax1.plot(time_periods, DA_p_actual, label='DA Price (Actual)', linewidth=2, color='#1f77b4', marker='o', markersize=3)
ax1.plot(time_periods, DA_p_f1, label='DA Price (Forecast 1)', linewidth=2, color='#ff7f0e', linestyle='--', alpha=0.7)
ax1.axhline(y=0, color='red', linestyle=':', linewidth=1, alpha=0.5, label='Zero Price Level')
ax1.set_xlabel('Quarter-Hour Period', fontsize=12)
ax1.set_ylabel('Price (EUR/MWh)', fontsize=12)
ax1.set_title('Day-Ahead Market Prices', fontsize=14, fontweight='bold')
ax1.legend(loc='best', fontsize=10)
ax1.grid(True, alpha=0.3)
ax1.set_xlim(0, len(time_periods) - 1)

# Subplot 2: Intraday Market Prices
ax2 = axes[1]
ax2.plot(time_periods, ID_p_actual, label='ID Price (Actual)', linewidth=2, color='#2ca02c', marker='s', markersize=3)
ax2.plot(time_periods, ID_p_f1, label='ID Price (Forecast 1)', linewidth=2, color='#d62728', linestyle='--', alpha=0.7)
ax2.axhline(y=0, color='red', linestyle=':', linewidth=1, alpha=0.5, label='Zero Price Level')
ax2.set_xlabel('Quarter-Hour Period', fontsize=12)
ax2.set_ylabel('Price (EUR/MWh)', fontsize=12)
ax2.set_title('Intraday Market Prices', fontsize=14, fontweight='bold')
ax2.legend(loc='best', fontsize=10)
ax2.grid(True, alpha=0.3)
ax2.set_xlim(0, len(time_periods) - 1)

plt.tight_layout()
plt.show()

# Print price statistics
print("\n" + "="*70)
print("MARKET PRICE STATISTICS")
print("="*70)

print("\n📊 DAY-AHEAD MARKET PRICES")
print(f"  Actual:    Min={DA_p_actual.min():.2f}, Max={DA_p_actual.max():.2f}, Mean={DA_p_actual.mean():.2f} EUR/MWh")
print(f"  Forecast:  Min={DA_p_f1.min():.2f}, Max={DA_p_f1.max():.2f}, Mean={DA_p_f1.mean():.2f} EUR/MWh")
print(f"  Forecast Error (Mean): {(DA_p_f1.mean() - DA_p_actual.mean()):.2f} EUR/MWh")

print("\n📊 INTRADAY MARKET PRICES")
print(f"  Actual:    Min={ID_p_actual.min():.2f}, Max={ID_p_actual.max():.2f}, Mean={ID_p_actual.mean():.2f} EUR/MWh")
print(f"  Forecast:  Min={ID_p_f1.min():.2f}, Max={ID_p_f1.max():.2f}, Mean={ID_p_f1.mean():.2f} EUR/MWh")
print(f"  Forecast Error (Mean): {(ID_p_f1.mean() - ID_p_actual.mean()):.2f} EUR/MWh")

print("\n📊 PRICE SPREAD (DA vs ID)")
print(f"  Actual Spread:    Mean={((DA_p_actual - ID_p_actual).mean()):.2f} EUR/MWh")
print(f"  Forecast Spread:  Mean={((DA_p_f1 - ID_p_f1).mean()):.2f} EUR/MWh")

# Count negative price periods
neg_da_actual = np.sum(DA_p_actual < 0)
neg_da_forecast = np.sum(DA_p_f1 < 0)
neg_id_actual = np.sum(ID_p_actual < 0)
neg_id_forecast = np.sum(ID_p_f1 < 0)

print("\n📊 NEGATIVE PRICE PERIODS")
print(f"  DA Actual:    {neg_da_actual} periods ({neg_da_actual/96*100:.1f}%)")
print(f"  DA Forecast:  {neg_da_forecast} periods ({neg_da_forecast/96*100:.1f}%)")
print(f"  ID Actual:    {neg_id_actual} periods ({neg_id_actual/96*100:.1f}%)")
print(f"  ID Forecast:  {neg_id_forecast} periods ({neg_id_forecast/96*100:.1f}%)")
print("="*70)

### 6.6 Market Prices vs Trading Volumes

In [ ]:
# Create triple-axis plots showing production forecast, DA and ID markets
fig, (ax0, ax1, ax2) = plt.subplots(3, 1, figsize=(14, 14))
time_periods = np.arange(len(Generation_f1))

# ===== SUBPLOT 0: Production Forecast vs Actual =====
ax0.plot(time_periods, Generation_f1, color='#ff7f0e', linewidth=2.5, linestyle='--', label='Generation Forecast', zorder=2)
ax0.plot(time_periods, Generation_actual, color='#1f77b4', linewidth=2.5, marker='o', markersize=3, label='Generation Actual', zorder=3)
ax0.fill_between(time_periods, Generation_f1, Generation_actual, alpha=0.2, color='gray', label='Forecast Error')
ax0.set_xlabel('Quarter-Hour Period', fontsize=12)
ax0.set_ylabel('Generation (MW)', fontsize=12)
ax0.set_title('Solar Generation: Forecast vs Actual', fontsize=14, fontweight='bold')
ax0.legend(loc='upper right', fontsize=10)
ax0.grid(True, alpha=0.3)
ax0.set_xlim(0, len(time_periods) - 1)

# ===== SUBPLOT 1: DA Market - Price and Volume =====
color_price = '#1f77b4'
color_price_forecast = '#ff7f0e'
color_volume = '#ffcc99'

# Plot DA price on left y-axis
ax1_left = ax1
ax1_left.plot(time_periods, DA_p_actual, color=color_price, linewidth=2.5, marker='o', markersize=3, label='DA Price (Actual)', zorder=3)
ax1_left.plot(time_periods, DA_p_f1, color=color_price_forecast, linewidth=2, linestyle='--', alpha=0.7, label='DA Price (Forecast)', zorder=2)
ax1_left.axhline(y=0, color='red', linestyle=':', linewidth=1, alpha=0.5)
ax1_left.set_xlabel('Quarter-Hour Period', fontsize=12)
ax1_left.set_ylabel('DA Price (EUR/MWh)', fontsize=12, color=color_price)
ax1_left.tick_params(axis='y', labelcolor=color_price)
ax1_left.set_title('Day-Ahead Market: Price vs Traded Volume', fontsize=14, fontweight='bold')
ax1_left.grid(True, alpha=0.3)
ax1_left.set_xlim(0, len(time_periods) - 1)

# Plot DA volume on right y-axis
ax1_right = ax1_left.twinx()
ax1_right.bar(time_periods, DA_stage2_results, color=color_volume, alpha=0.5, width=0.8, label='DA Traded (MW)', zorder=1)
ax1_right.set_ylabel('DA Traded Volume (MW)', fontsize=12, color='#cc6600')
ax1_right.tick_params(axis='y', labelcolor='#cc6600')

# Combine legends
lines1, labels1 = ax1_left.get_legend_handles_labels()
lines2, labels2 = ax1_right.get_legend_handles_labels()
ax1_left.legend(lines1 + lines2, labels1 + labels2, loc='upper left', fontsize=10)

# ===== SUBPLOT 2: ID Market - Price and Volume =====
color_price_id = '#2ca02c'
color_price_id_forecast = '#d62728'
color_volume_id = '#ffb3ba'

# Plot ID price on left y-axis
ax2_left = ax2
ax2_left.plot(time_periods, ID_p_actual, color=color_price_id, linewidth=2.5, marker='s', markersize=3, label='ID Price (Actual)', zorder=3)
ax2_left.plot(time_periods, ID_p_f1, color=color_price_id_forecast, linewidth=2, linestyle='--', alpha=0.7, label='ID Price (Forecast)', zorder=2)
ax2_left.axhline(y=0, color='red', linestyle=':', linewidth=1, alpha=0.5)
ax2_left.set_xlabel('Quarter-Hour Period', fontsize=12)
ax2_left.set_ylabel('ID Price (EUR/MWh)', fontsize=12, color=color_price_id)
ax2_left.tick_params(axis='y', labelcolor=color_price_id)
ax2_left.set_title('Intraday Market: Price vs Traded Volume', fontsize=14, fontweight='bold')
ax2_left.grid(True, alpha=0.3)
ax2_left.set_xlim(0, len(time_periods) - 1)

# Plot ID volume on right y-axis (showing net position from Stage 3)
ax2_right = ax2_left.twinx()
ID_net = np.array(ID_stage3_results)
# Color bars based on buy (positive) or sell (negative)
colors_id = ['#ff6b6b' if x > 0 else '#9467bd' for x in ID_net]
ax2_right.bar(time_periods, ID_net, color=colors_id, alpha=0.5, width=0.8, label='ID Traded (MW)', zorder=1)
ax2_right.axhline(y=0, color='black', linestyle='-', linewidth=0.5, alpha=0.3)
ax2_right.set_ylabel('ID Traded Volume (MW)', fontsize=12, color='#cc0066')
ax2_right.tick_params(axis='y', labelcolor='#cc0066')

# Combine legends with color explanation
from matplotlib.patches import Patch
lines1, labels1 = ax2_left.get_legend_handles_labels()
legend_elements = lines1 + [
    Patch(facecolor='#ff6b6b', alpha=0.5, label='ID Buy (Positive)'),
    Patch(facecolor='#9467bd', alpha=0.5, label='ID Sell (Negative)')
]
ax2_left.legend(handles=legend_elements, loc='upper left', fontsize=10)

plt.tight_layout()
plt.show()

# Print summary statistics
print("\n" + "="*70)
print("PRICE vs VOLUME ANALYSIS")
print("="*70)

print("\n📊 GENERATION FORECAST vs ACTUAL")
print(f"  Forecast Total:   {Generation_f1.sum()/4:.2f} MWh")
print(f"  Actual Total:     {Generation_actual.sum()/4:.2f} MWh")
print(f"  Forecast Error:   {(Generation_f1.sum() - Generation_actual.sum())/4:.2f} MWh ({((Generation_f1.sum() - Generation_actual.sum())/Generation_f1.sum())*100:.1f}%)")
print(f"  Forecast Avg:     {Generation_f1.mean():.2f} MW")
print(f"  Actual Avg:       {Generation_actual.mean():.2f} MW")

print("\n📊 DAY-AHEAD MARKET")
print(f"  Total DA Volume:  {sum(DA_stage2_results):.2f} MW (cumulative)")
print(f"  Avg DA Volume:    {np.mean(DA_stage2_results):.2f} MW per period")
print(f"  Avg DA Price:     {DA_p_actual.mean():.2f} EUR/MWh")
print(f"  Total DA Revenue: {sum([DA_stage2_results[t] * DA_p_actual[t]/4 for t in range(96)]):.2f} EUR")

print("\n📊 INTRADAY MARKET")
id_buys_count = sum(1 for x in ID_net if x > 0)
id_sells_count = sum(1 for x in ID_net if x < 0)
id_buy_volume = sum(x for x in ID_net if x > 0)
id_sell_volume = sum(abs(x) for x in ID_net if x < 0)

print(f"  Buy Periods:      {id_buys_count} periods, {id_buy_volume:.2f} MW total")
print(f"  Sell Periods:     {id_sells_count} periods, {id_sell_volume:.2f} MW total")
print(f"  Net ID Position:  {sum(ID_net):.2f} MW")
print(f"  Avg ID Price:     {ID_p_actual.mean():.2f} EUR/MWh")

# Calculate ID market cost/revenue
id_revenue = sum([ID_stage3_results[t] * ID_p_actual[t]/4 for t in range(96)])
print(f"  Total ID Revenue: {id_revenue:.2f} EUR")
print("="*70)

### Day-Ahead Price Comparison Visualization

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# Extract DA prices for October 4 and 5 only
print("=== ACTUAL DA Prices for October 4-5, 2025 ===")
print("Oct 4: Forecast Source Day | Oct 5: Analysis Day")
print("Using ACTUAL DA prices from DA_p_full (already processed and sorted)\n")

# Use the already-processed DA_p_full which contains actual DA prices
# (This was created earlier in Section 4 with proper sorting and processing)

# Filter for October 4-5 actual dates
oct_4_start = pd.Timestamp("2025-10-04 00:00:00").tz_localize("Europe/Berlin")
oct_5_end = pd.Timestamp("2025-10-05 23:45:00").tz_localize("Europe/Berlin")

# Get data for Oct 4-5 actual dates
oct_3_6_data = DA_p_full[
    (DA_p_full['PeriodStartLocal'] >= oct_4_start) & (DA_p_full['PeriodStartLocal'] <= oct_5_end)
].copy()

# Add date column for grouping
oct_3_6_data['Date'] = oct_3_6_data['PeriodStartLocal'].dt.date
oct_3_6_data['Time'] = oct_3_6_data['PeriodStartLocal'].dt.time

# Create figure with two subplots
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 12))

# Plot 1: Line chart showing Oct 4 and Oct 5
colors_days = ['#A23B72', '#F18F01']  # Purple for Oct 4, Yellow for Oct 5
day_names = ['Oct 4 (Fri)', 'Oct 5 (Sat)']
unique_dates = sorted(oct_3_6_data['Date'].unique())

for idx, date in enumerate(unique_dates):
    day_data = oct_3_6_data[oct_3_6_data['Date'] == date].copy()
    day_data['Period'] = range(len(day_data))
    
    ax1.plot(day_data['Period'], day_data['DA_price'], 
             color=colors_days[idx], linewidth=2, marker='o', markersize=3,
             label=f'{day_names[idx]} (Mean: {day_data["DA_price"].mean():.2f} €/MWh)', 
             alpha=0.8)

ax1.set_xlabel('Quarter-Hour Period (0-95)', fontsize=12, fontweight='bold')
ax1.set_ylabel('DA Price (EUR/MWh)', fontsize=12, fontweight='bold')
ax1.set_title('Day-Ahead ACTUAL Price Comparison: Oct 4-5, 2025', fontsize=14, fontweight='bold', pad=20)
ax1.legend(loc='best', fontsize=10, framealpha=0.9)
ax1.grid(True, alpha=0.3, linestyle='--')
ax1.axhline(y=0, color='black', linestyle='-', linewidth=0.8, alpha=0.5)

# Highlight key days
ax1.text(0.02, 0.98, '← Oct 4: Forecast Source', transform=ax1.transAxes,
         fontsize=10, verticalalignment='top', bbox=dict(boxstyle='round', 
         facecolor='#A23B72', alpha=0.3, edgecolor='#A23B72'))
ax1.text(0.02, 0.92, '← Oct 5: Analysis Day', transform=ax1.transAxes,
         fontsize=10, verticalalignment='top', bbox=dict(boxstyle='round', 
         facecolor='#F18F01', alpha=0.3, edgecolor='#F18F01'))

# Plot 2: Heatmap-style view
heatmap_data = oct_3_6_data.pivot_table(
    index='Time', 
    columns='Date', 
    values='DA_price'
)

im = ax2.imshow(heatmap_data.T.values, aspect='auto', cmap='RdYlGn_r', 
                interpolation='nearest', vmin=-10, vmax=150)

ax2.set_yticks(range(len(unique_dates)))
ax2.set_yticklabels([day_names[i] for i in range(len(unique_dates))], fontsize=11)
ax2.set_xticks(range(0, 96, 6))
ax2.set_xticklabels([f'{i//4:02d}:00' for i in range(0, 96, 6)], fontsize=9)
ax2.set_xlabel('Time of Day', fontsize=12, fontweight='bold')
ax2.set_ylabel('Date', fontsize=12, fontweight='bold')
ax2.set_title('DA ACTUAL Price Heatmap: Oct 4-5, 2025', fontsize=14, fontweight='bold', pad=20)

# Add colorbar
cbar = plt.colorbar(im, ax=ax2)
cbar.set_label('DA Price (EUR/MWh)', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

# Print summary statistics
print("\n=== Summary Statistics by Day (ACTUAL DA Prices) ===\n")
for idx, date in enumerate(unique_dates):
    day_data = oct_3_6_data[oct_3_6_data['Date'] == date]
    print(f"{day_names[idx]}:")
    print(f"  Mean:   {day_data['DA_price'].mean():8.2f} EUR/MWh")
    print(f"  Median: {day_data['DA_price'].median():8.2f} EUR/MWh")
    print(f"  Min:    {day_data['DA_price'].min():8.2f} EUR/MWh")
    print(f"  Max:    {day_data['DA_price'].max():8.2f} EUR/MWh")
    print(f"  Std:    {day_data['DA_price'].std():8.2f} EUR/MWh")
    print(f"  Negative price periods: {(day_data['DA_price'] < 0).sum()}/96")
    print()

In [ ]:
# Create detailed comparison table
print("\n=== Detailed DA Price Comparison Table ===")
print("(Showing key hours: midnight, 6am, noon, 6pm, midnight)\n")

# Select key time periods
key_hours = [0, 24, 48, 72, 95]  # Midnight, 6am, noon, 6pm, last period

comparison_table = []
for period_idx in key_hours:
    row_data = {'Period': period_idx, 'Time': f"{(period_idx//4):02d}:{(period_idx%4)*15:02d}"}
    
    for idx, date in enumerate(unique_dates):
        day_data = oct_3_6_data[oct_3_6_data['Date'] == date].reset_index(drop=True)
        if period_idx < len(day_data):
            row_data[day_names[idx]] = f"{day_data.loc[period_idx, 'DA_price']:.2f}"
        else:
            row_data[day_names[idx]] = "N/A"
    
    comparison_table.append(row_data)

comparison_df = pd.DataFrame(comparison_table)
print(comparison_df.to_string(index=False))

# Create full detailed table (showing all 96 periods)
print("\n\n=== Full DA Price Table (All 96 Periods) ===")
print("Prices in EUR/MWh\n")

# Pivot the data to show all days side by side
full_table = oct_3_6_data.pivot_table(
    index=oct_3_6_data['PeriodStartLocal'].dt.time,
    columns='Date',
    values='DA_price',
    aggfunc='first'
)

# Rename columns to be more readable
full_table.columns = [day_names[i] for i in range(len(unique_dates))]
full_table.index.name = 'Time'

# Display first 20 and last 10 rows
print("First 20 periods (00:00 - 04:45):")
print(full_table.head(20).to_string())

print("\n... (periods 21-75 omitted for brevity) ...\n")

print("Last 20 periods (19:00 - 23:45):")
print(full_table.tail(20).to_string())

# Export full table
full_table_reset = full_table.reset_index()
print(f"\n\nFull table shape: {full_table_reset.shape}")
print(f"Total periods per day: {len(full_table_reset)}")

# Show the full table
full_table_reset

## 7. Battery Evaluation

First metric: curtailment rate

In [ ]:
# curtailed_energy_mwh is the physical curtailment = possible generation - delivered generation
curtailment_rate = (curtailed_energy_mwh / total_possible_energy_mwh) if total_possible_energy_mwh > 0 else np.nan

print(f"  Curtailed Energy:         {curtailed_energy_mwh:.2f} MWh")
print(f"  Curtailment Rate:         {curtailment_rate:.2%}")

Second metric: temporal concentration

In [ ]:
# Curtailment per period (MWh): possible generation minus delivered generation
curtailment_mwh_period = (
    (results_df["Generation_Possible_MW"] - results_df["Generation_Delivered_MW"]) / 4
).clip(lower=0)

# Number of periods to include in the top 10% (at least 1)
n_periods = len(curtailment_mwh_period)
top_n = max(1, int(np.ceil(0.10 * n_periods)))

# Share of total curtailment concentrated in the top 10% periods
total_curtailment = curtailment_mwh_period.sum()
top_10_sum = curtailment_mwh_period.nlargest(top_n).sum()
temporal_concentration = (top_10_sum / total_curtailment) if total_curtailment > 0 else np.nan

print(f"Temporal Concentration = {temporal_concentration:.4f}")

In [ ]:
# --- 1) Curtailment per 15-min period (MWh) ---
plot_df = results_df.copy()
plot_df["PeriodStartLocal"] = pd.to_datetime(plot_df["PeriodStartLocal"])
plot_df["curtailment_mwh"] = (
    (plot_df["Generation_Possible_MW"] - plot_df["Generation_Delivered_MW"]) / 4
).clip(lower=0)

# --- 2) Identify top 10% highest-curtailment periods ---
n = len(plot_df)
top_n = max(1, int(np.ceil(0.10 * n)))
top_idx = plot_df["curtailment_mwh"].nlargest(top_n).index
plot_df["is_top_10pct"] = False
plot_df.loc[top_idx, "is_top_10pct"] = True

# --- 3) Plot time series, highlighting top concentration periods ---
x = np.arange(n)
colors = np.where(plot_df["is_top_10pct"], "#d62728", "#1f77b4")

plt.figure(figsize=(16, 5))
plt.bar(x, plot_df["curtailment_mwh"], color=colors, width=0.85, alpha=0.9)

# Optional line on top for readability
plt.plot(x, plot_df["curtailment_mwh"], color="black", linewidth=0.8, alpha=0.5)

# Label x-axis hourly (every 4 periods)
hour_ticks = np.arange(0, n, 4)
hour_labels = plot_df["PeriodStartLocal"].dt.strftime("%H:%M").iloc[hour_ticks]
plt.xticks(hour_ticks, hour_labels, rotation=45)

plt.title("Curtailment by 15-min Period (Top 10% Highlighted)")
plt.xlabel("Time of day")
plt.ylabel("Curtailment (MWh per 15-min period)")
plt.grid(axis="y", alpha=0.3)

# Legend
from matplotlib.patches import Patch
legend_handles = [
    Patch(facecolor="#d62728", label="Top 10% curtailment periods"),
    Patch(facecolor="#1f77b4", label="Other periods"),
]
plt.legend(handles=legend_handles, loc="upper right")

plt.tight_layout()
plt.show()

Evaluation of battery storage relevance based on curtailment rate and WP7 studies

In [ ]:
# Battery storage relevance assessment based on your threshold table
if np.isnan(curtailment_rate):
    relevance = "N/A"
    interpretation = "No production data available for the selected horizon."
elif curtailment_rate < 0.02:
    relevance = "Low"
    interpretation = "Curtailment is in the <1-2% range, typically indicating sufficient flexibility."
elif curtailment_rate <= 0.04:
    if np.isnan(temporal_concentration):
        relevance = "Moderate"
        interpretation = "Curtailment is in the 1-4% range; storage relevance is moderate."
    elif temporal_concentration >= 0.50:
        relevance = "Moderate to High"
        interpretation = "Curtailment is 1-4% but highly concentrated in few periods, increasing storage value."
    else:
        relevance = "Moderate"
        interpretation = "Curtailment is 1-4% and not strongly concentrated."
elif curtailment_rate > 0.05:
    relevance = "High"
    interpretation = "Curtailment >5% suggests persistent constraints; storage is highly relevant."
else:
    relevance = "Moderate to High"
    interpretation = "Curtailment is between 4-5%; storage relevance is elevated."
print(f"Battery Storage Relevance = {relevance}")
print(f"Evaluation: {interpretation}")

## 8. Evalutation and Visualization

In [ ]:
# ============================================
# Revenue + Energy by channel (DA / ID / Subsidies / aFRR)
# Uses existing `results_df` from your notebook
# ============================================
import numpy as np
import pandas as pd

channel_data = results_df.copy()

# --- Basic energy terms (MWh per 15-min period) ---
channel_data["Produced_MWh"]  = channel_data["Generation_Possible_MW"] / 4
channel_data["Delivered_MWh"] = channel_data["Generation_Delivered_MW"] / 4
channel_data["Curtailed_MWh"] = ((channel_data["Generation_Possible_MW"] - channel_data["Generation_Delivered_MW"]) / 4).clip(lower=0)

# --- DA channel ---
channel_data["DA_Sold_MWh"]    = channel_data["DA_MW"] / 4
channel_data["DA_Revenue_EUR"] = channel_data["DA_MW"] / 4 * channel_data["DA_Price_EUR_MWh"]

# --- ID channel (split into sell/buy) ---
channel_data["ID_Sell_MWh"]          = channel_data["ID_MW"].clip(lower=0) / 4
channel_data["ID_Buy_MWh"]           = (-channel_data["ID_MW"].clip(upper=0)) / 4
channel_data["ID_Net_MWh"]           = channel_data["ID_MW"] / 4
channel_data["ID_Sell_Revenue_EUR"]  = channel_data["ID_Sell_MWh"] * channel_data["ID_Price_EUR_MWh"]
channel_data["ID_Buy_Cost_EUR"]      = channel_data["ID_Buy_MWh"]  * channel_data["ID_Price_EUR_MWh"]
channel_data["ID_Net_Revenue_EUR"]   = channel_data["ID_Sell_Revenue_EUR"] - channel_data["ID_Buy_Cost_EUR"]

# --- Subsidy channel (EEG / market premium) ---
# already EUR per period in your notebook
if "Market premium_EUR" in channel_data.columns:
    channel_data["Subsidy_Revenue_EUR"] = channel_data["Market premium_EUR"]
else:
    channel_data["Subsidy_Revenue_EUR"] = 0.0

# --- aFRR channel (if available in results_df) ---
# Try to auto-detect columns, otherwise fallback to zero
afrr_energy_col = None
afrr_revenue_col = None

for c in channel_data.columns:
    lc = c.lower()
    if afrr_energy_col is None and ("afrr" in lc or "eam" in lc or "bcap" in lc) and ("mwh" in lc or "mw" in lc or "energy" in lc):
        afrr_energy_col = c
    if afrr_revenue_col is None and ("afrr" in lc or "eam" in lc or "bcap" in lc) and ("eur" in lc or "revenue" in lc):
        afrr_revenue_col = c

if afrr_energy_col:
    if "mw" in afrr_energy_col.lower() and "mwh" not in afrr_energy_col.lower():
        channel_data["aFRR_Energy_MWh"] = channel_data[afrr_energy_col] / 4
    else:
        channel_data["aFRR_Energy_MWh"] = channel_data[afrr_energy_col]
else:
    channel_data["aFRR_Energy_MWh"] = 0.0

channel_data["aFRR_Revenue_EUR"] = channel_data[afrr_revenue_col] if afrr_revenue_col else 0.0

# --- Optional: imbalance settlement ---
channel_data["Imbalance_Settlement_EUR"] = (
    channel_data["Imb_pos_MW"] / 4 * channel_data["Imb_price"] - channel_data["Imb_neg_MW"] / 4 * channel_data["Imb_price"]
)

# ============================================
# Summary table
# ============================================
summary = pd.DataFrame({
    "Channel": [
        "Production (possible)",
        "Delivered (actual)",
        "Curtailment",
        "DA",
        "ID (sell)",
        "ID (buy)",
        "ID (net)",
        "Subsidies",
        "aFRR",
        "Imbalance settlement",
        "TOTAL (DA + ID net + Subsidies + aFRR + Imbalance)"
    ],
    "Energy_MWh": [
        channel_data["Produced_MWh"].sum(),
        channel_data["Delivered_MWh"].sum(),
        channel_data["Curtailed_MWh"].sum(),
        channel_data["DA_Sold_MWh"].sum(),
        channel_data["ID_Sell_MWh"].sum(),
        channel_data["ID_Buy_MWh"].sum(),
        channel_data["ID_Net_MWh"].sum(),
        np.nan,  # subsidies are not energy channel
        channel_data["aFRR_Energy_MWh"].sum(),
        np.nan,  # imbalance settlement is not energy channel
        np.nan
    ],
    "Revenue_EUR": [
        np.nan,  # production itself has no direct revenue
        np.nan,  # delivered itself has no direct revenue
        np.nan,  # curtailment is opportunity loss, not booked revenue
        channel_data["DA_Revenue_EUR"].sum(),
        channel_data["ID_Sell_Revenue_EUR"].sum(),
        -channel_data["ID_Buy_Cost_EUR"].sum(),
        channel_data["ID_Net_Revenue_EUR"].sum(),
        channel_data["Subsidy_Revenue_EUR"].sum(),
        channel_data["aFRR_Revenue_EUR"].sum(),
        channel_data["Imbalance_Settlement_EUR"].sum(),
        (
            channel_data["DA_Revenue_EUR"].sum()
            + channel_data["ID_Net_Revenue_EUR"].sum()
            + channel_data["Subsidy_Revenue_EUR"].sum()
            + channel_data["aFRR_Revenue_EUR"].sum()
            + channel_data["Imbalance_Settlement_EUR"].sum()
        )
    ]
})

# Pretty print
pd.options.display.float_format = "{:,.2f}".format
print("\n=== ENERGY & REVENUE BY CHANNEL ===")
print(summary.to_string(index=False))

# Quick diagnostics for aFRR auto-detection
print("\n[aFRR detection]")
print(f"Energy column used : {afrr_energy_col if afrr_energy_col else 'None (set to 0)'}")
print(f"Revenue column used: {afrr_revenue_col if afrr_revenue_col else 'None (set to 0)'}")

In [ ]:
# ============================================================
# Monthly + Hourly summary statistics (energy & revenue channels)
# Uses `results_df` from your notebook
# ============================================================
import numpy as np
import pandas as pd

summary_data = results_df.copy()

# ---------- Time fields ----------
summary_data["PeriodStartLocal"] = pd.to_datetime(summary_data["PeriodStartLocal"])
summary_data["month"] = summary_data["PeriodStartLocal"].dt.to_period("M").astype(str)   # e.g. 2025-10
summary_data["hour"]  = summary_data["PeriodStartLocal"].dt.hour                          # 0..23

# ---------- Core energy terms ----------
summary_data["Produced_MWh"]  = summary_data["Generation_Possible_MW"] / 4
summary_data["Delivered_MWh"] = summary_data["Generation_Delivered_MW"] / 4
summary_data["Curtailed_MWh"] = ((summary_data["Generation_Possible_MW"] - summary_data["Generation_Delivered_MW"]) / 4).clip(lower=0)

# ---------- DA ----------
summary_data["DA_Sold_MWh"]    = summary_data["DA_MW"] / 4
summary_data["DA_Revenue_EUR"] = summary_data["DA_MW"] / 4 * summary_data["DA_Price_EUR_MWh"]

# ---------- ID (split) ----------
summary_data["ID_Sell_MWh"]         = summary_data["ID_MW"].clip(lower=0) / 4
summary_data["ID_Buy_MWh"]          = (-summary_data["ID_MW"].clip(upper=0)) / 4
summary_data["ID_Net_MWh"]          = summary_data["ID_MW"] / 4
summary_data["ID_Sell_Revenue_EUR"] = summary_data["ID_Sell_MWh"] * summary_data["ID_Price_EUR_MWh"]
summary_data["ID_Buy_Cost_EUR"]     = summary_data["ID_Buy_MWh"]  * summary_data["ID_Price_EUR_MWh"]
summary_data["ID_Net_Revenue_EUR"]  = summary_data["ID_Sell_Revenue_EUR"] - summary_data["ID_Buy_Cost_EUR"]

# ---------- Subsidies ----------
summary_data["Subsidy_Revenue_EUR"] = summary_data["Market premium_EUR"] if "Market premium_EUR" in summary_data.columns else 0.0

# ---------- Imbalance settlement ----------
summary_data["Imbalance_Settlement_EUR"] = (
    summary_data["Imb_pos_MW"] / 4 * summary_data["Imb_price"] - summary_data["Imb_neg_MW"] / 4 * summary_data["Imb_price"]
)

# ---------- aFRR (optional autodetect) ----------
afrr_energy_col, afrr_revenue_col = None, None
for c in summary_data.columns:
    lc = c.lower()
    if afrr_energy_col is None and ("afrr" in lc or "eam" in lc or "bcap" in lc) and ("mwh" in lc or "mw" in lc or "energy" in lc):
        afrr_energy_col = c
    if afrr_revenue_col is None and ("afrr" in lc or "eam" in lc or "bcap" in lc) and ("eur" in lc or "revenue" in lc):
        afrr_revenue_col = c

if afrr_energy_col:
    if "mw" in afrr_energy_col.lower() and "mwh" not in afrr_energy_col.lower():
        summary_data["aFRR_Energy_MWh"] = summary_data[afrr_energy_col] / 4
    else:
        summary_data["aFRR_Energy_MWh"] = summary_data[afrr_energy_col]
else:
    summary_data["aFRR_Energy_MWh"] = 0.0

summary_data["aFRR_Revenue_EUR"] = summary_data[afrr_revenue_col] if afrr_revenue_col else 0.0

# ---------- Total revenue ----------
summary_data["Total_Revenue_EUR"] = (
    summary_data["DA_Revenue_EUR"]
    + summary_data["ID_Net_Revenue_EUR"]
    + summary_data["Subsidy_Revenue_EUR"]
    + summary_data["aFRR_Revenue_EUR"]
    + summary_data["Imbalance_Settlement_EUR"]
)

# ============================================================
# Helper to build grouped summary
# ============================================================
def build_summary(group_col: str) -> pd.DataFrame:
    g = summary_data.groupby(group_col, dropna=False).agg(
        Periods=("PeriodStartLocal", "count"),
        Produced_MWh=("Produced_MWh", "sum"),
        Delivered_MWh=("Delivered_MWh", "sum"),
        Curtailed_MWh=("Curtailed_MWh", "sum"),
        Curtailment_Rate_pct=("Curtailed_MWh", lambda s: 100 * s.sum() / max(summary_data.loc[s.index, "Produced_MWh"].sum(), 1e-9)),
        DA_Sold_MWh=("DA_Sold_MWh", "sum"),
        DA_Revenue_EUR=("DA_Revenue_EUR", "sum"),
        ID_Sell_MWh=("ID_Sell_MWh", "sum"),
        ID_Buy_MWh=("ID_Buy_MWh", "sum"),
        ID_Net_MWh=("ID_Net_MWh", "sum"),
        ID_Net_Revenue_EUR=("ID_Net_Revenue_EUR", "sum"),
        Subsidy_Revenue_EUR=("Subsidy_Revenue_EUR", "sum"),
        aFRR_Energy_MWh=("aFRR_Energy_MWh", "sum"),
        aFRR_Revenue_EUR=("aFRR_Revenue_EUR", "sum"),
        Imbalance_Settlement_EUR=("Imbalance_Settlement_EUR", "sum"),
        Total_Revenue_EUR=("Total_Revenue_EUR", "sum"),
        DA_Price_Mean=("DA_Price_EUR_MWh", "mean"),
        ID_Price_Mean=("ID_Price_EUR_MWh", "mean"),
        Imb_Price_Mean=("Imb_price", "mean"),
    ).reset_index()

    # Additional intensity KPIs
    g["Revenue_per_Produced_EUR_per_MWh"] = g["Total_Revenue_EUR"] / g["Produced_MWh"].replace(0, np.nan)
    g["Revenue_per_Delivered_EUR_per_MWh"] = g["Total_Revenue_EUR"] / g["Delivered_MWh"].replace(0, np.nan)
    return g

# ============================================================
# 1) Monthly summary
# ============================================================
monthly_summary = build_summary("month").sort_values("month").reset_index(drop=True)

# ============================================================
# 2) Hourly summary (across all days)
# ============================================================
hourly_summary = build_summary("hour").sort_values("hour").reset_index(drop=True)

# Optional: also monthly-hour matrix for quick profile
monthly_hourly = (
    summary_data.groupby(["month", "hour"], dropna=False)
      .agg(
          Produced_MWh=("Produced_MWh", "sum"),
          Delivered_MWh=("Delivered_MWh", "sum"),
          Curtailed_MWh=("Curtailed_MWh", "sum"),
          DA_Revenue_EUR=("DA_Revenue_EUR", "sum"),
          ID_Net_Revenue_EUR=("ID_Net_Revenue_EUR", "sum"),
          Subsidy_Revenue_EUR=("Subsidy_Revenue_EUR", "sum"),
          aFRR_Revenue_EUR=("aFRR_Revenue_EUR", "sum"),
          Imbalance_Settlement_EUR=("Imbalance_Settlement_EUR", "sum"),
          Total_Revenue_EUR=("Total_Revenue_EUR", "sum"),
      )
      .reset_index()
      .sort_values(["month", "hour"])
)

# ============================================================
# Display
# ============================================================
pd.options.display.float_format = "{:,.2f}".format

print("\n=== MONTHLY SUMMARY STATISTICS ===")
print(monthly_summary.to_string(index=False))

print("\n=== HOURLY SUMMARY STATISTICS (across all days) ===")
print(hourly_summary.to_string(index=False))

print("\n=== MONTH x HOUR SUMMARY (first 30 rows) ===")
print(monthly_hourly.head(30).to_string(index=False))

print("\n[aFRR detection]")
print(f"Energy column used : {afrr_energy_col if afrr_energy_col else 'None (set to 0)'}")
print(f"Revenue column used: {afrr_revenue_col if afrr_revenue_col else 'None (set to 0)'}")

In [ ]:
# ============================================
# Monthly stacked revenue bars by channel
# (DA / ID net / Subsidies / aFRR / Imbalance)
# Uses existing `results_df`
# ============================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rev_data = results_df.copy()
rev_data["PeriodStartLocal"] = pd.to_datetime(rev_data["PeriodStartLocal"])
rev_data["month"] = rev_data["PeriodStartLocal"].dt.to_period("M").astype(str)

# --- Revenue channels ---
rev_data["DA_Revenue_EUR"] = rev_data["DA_MW"] / 4 * rev_data["DA_Price_EUR_MWh"]

rev_data["ID_Sell_MWh"] = rev_data["ID_MW"].clip(lower=0) / 4
rev_data["ID_Buy_MWh"]  = (-rev_data["ID_MW"].clip(upper=0)) / 4
rev_data["ID_Sell_Revenue_EUR"] = rev_data["ID_Sell_MWh"] * rev_data["ID_Price_EUR_MWh"]
rev_data["ID_Buy_Cost_EUR"]     = rev_data["ID_Buy_MWh"]  * rev_data["ID_Price_EUR_MWh"]
rev_data["ID_Net_Revenue_EUR"]  = rev_data["ID_Sell_Revenue_EUR"] - rev_data["ID_Buy_Cost_EUR"]

rev_data["Subsidy_Revenue_EUR"] = rev_data["Market premium_EUR"] if "Market premium_EUR" in rev_data.columns else 0.0

rev_data["Imbalance_Settlement_EUR"] = (
    rev_data["Imb_pos_MW"] / 4 * rev_data["Imb_price"] - rev_data["Imb_neg_MW"] / 4 * rev_data["Imb_price"]
)

# --- aFRR revenue autodetect (optional) ---
afrr_revenue_col = None
for c in rev_data.columns:
    lc = c.lower()
    if ("afrr" in lc or "eam" in lc or "bcap" in lc) and ("eur" in lc or "revenue" in lc):
        afrr_revenue_col = c
        break
rev_data["aFRR_Revenue_EUR"] = rev_data[afrr_revenue_col] if afrr_revenue_col else 0.0

# Aggregate monthly
monthly_rev = rev_data.groupby("month", as_index=True).agg({
    "DA_Revenue_EUR": "sum",
    "ID_Net_Revenue_EUR": "sum",
    "Subsidy_Revenue_EUR": "sum",
    "aFRR_Revenue_EUR": "sum",
    "Imbalance_Settlement_EUR": "sum"
}).sort_index()

# Split positive/negative for clean stacked plotting
channels = [
    ("DA_Revenue_EUR", "DA"),
    ("ID_Net_Revenue_EUR", "ID (net)"),
    ("Subsidy_Revenue_EUR", "Subsidies"),
    ("aFRR_Revenue_EUR", "aFRR"),
    ("Imbalance_Settlement_EUR", "Imbalance"),
]
colors = {
    "DA": "#1f77b4",
    "ID (net)": "#ff7f0e",
    "Subsidies": "#2ca02c",
    "aFRR": "#9467bd",
    "Imbalance": "#d62728",
}

x = np.arange(len(monthly_rev.index))
fig, ax = plt.subplots(figsize=(14, 7))

pos_base = np.zeros(len(monthly_rev))
neg_base = np.zeros(len(monthly_rev))

for col, label in channels:
    vals = monthly_rev[col].values
    vals_pos = np.where(vals > 0, vals, 0.0)
    vals_neg = np.where(vals < 0, vals, 0.0)

    ax.bar(x, vals_pos, bottom=pos_base, color=colors[label], label=label)
    ax.bar(x, vals_neg, bottom=neg_base, color=colors[label])

    pos_base += vals_pos
    neg_base += vals_neg

# Total net revenue line
monthly_rev["Total_Net_Revenue_EUR"] = monthly_rev.sum(axis=1)
ax.plot(x, monthly_rev["Total_Net_Revenue_EUR"].values, color="black", marker="o", linewidth=2, label="Total net")

# Formatting
ax.axhline(0, color="black", linewidth=1)
ax.set_xticks(x)
ax.set_xticklabels(monthly_rev.index, rotation=45, ha="right")
ax.set_ylabel("Revenue (EUR)")
ax.set_xlabel("Month")
ax.set_title("Monthly Stacked Revenue by Channel")
ax.grid(axis="y", alpha=0.3)
ax.legend(loc="best")
plt.tight_layout()
plt.show()

# Optional table
print("\n=== Monthly Revenue Table (EUR) ===")
display_cols = ["DA_Revenue_EUR", "ID_Net_Revenue_EUR", "Subsidy_Revenue_EUR", "aFRR_Revenue_EUR", "Imbalance_Settlement_EUR", "Total_Net_Revenue_EUR"]
print(monthly_rev[display_cols].round(2).to_string())

print("\n[aFRR revenue column used]")
print(afrr_revenue_col if afrr_revenue_col else "None found (aFRR set to 0)")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np

# ── Revenue components ──────────────────────────────────────────────────────
opt_da_rev    = results_df['DA_Revenue_EUR'].sum()
opt_id_rev    = results_df['ID_Revenue_EUR'].sum()
opt_eeg_rev   = results_df['Market premium_EUR'].sum()
opt_imb_rev   = results_df['Imb_Settlement_EUR'].sum()
opt_total_rev = opt_da_rev + opt_id_rev + opt_eeg_rev + opt_imb_rev

bench_da_rev    = benchmark_DA_revenue
bench_eeg_rev   = benchmark_EEG_premium
bench_imb_rev   = benchmark_imb_settlement
bench_total_rev = bench_da_rev + bench_eeg_rev + bench_imb_rev

diff_total = opt_total_rev - bench_total_rev
diff_pct   = diff_total / abs(bench_total_rev) * 100 if bench_total_rev != 0 else 0

# ── Energy ──────────────────────────────────────────────────────────────────
opt_energy_mwh   = (np.array(DA_stage3_results) + np.array(ID_stage3_results)).sum() / 4
bench_energy_mwh = np.array(benchmark_DA).sum() / 4
actual_energy_mwh = np.array(Generation_actual).sum() / 4

# ── Summary table ───────────────────────────────────────────────────────────
summary = pd.DataFrame({
    'Strategy': [
        'No-Curtailment Optimised\n(DA + ID hedging)',
        'Benchmark\n(Naive DA-only)',
        'Difference\n(Opt − Bench)'
    ],
    'DA Revenue (EUR)': [opt_da_rev, bench_da_rev, opt_da_rev - bench_da_rev],
    'ID Revenue (EUR)': [opt_id_rev, 0.0, opt_id_rev],
    'EEG Premium (EUR)': [opt_eeg_rev, bench_eeg_rev, opt_eeg_rev - bench_eeg_rev],
    'Imbalance (EUR)': [opt_imb_rev, bench_imb_rev, opt_imb_rev - bench_imb_rev],
    'TOTAL Revenue (EUR)': [opt_total_rev, bench_total_rev, diff_total],
    'Energy Sold (MWh)': [opt_energy_mwh, bench_energy_mwh, opt_energy_mwh - bench_energy_mwh],
})

# ── Bar chart ────────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(15, 10))
gs  = gridspec.GridSpec(2, 2, figure=fig, hspace=0.45, wspace=0.35)

strategies   = ['Optimised\n(DA+ID)', 'Benchmark\n(DA-only)']
colors_strat = ['darkgreen', 'darkorange']

# 1) Stacked revenue bar
ax_bar = fig.add_subplot(gs[0, :])
components = {
    'DA Revenue':    [opt_da_rev,  bench_da_rev],
    'ID Revenue':    [opt_id_rev,  0.0],
    'EEG Premium':   [opt_eeg_rev, bench_eeg_rev],
    'Imbalance':     [opt_imb_rev, bench_imb_rev],
}
comp_colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']
x = np.arange(2)
width = 0.45
bottoms = np.zeros(2)
for (name, vals), col in zip(components.items(), comp_colors):
    bars = ax_bar.bar(x, vals, width, bottom=bottoms, label=name, color=col, alpha=0.85)
    for bar, v in zip(bars, vals):
        if abs(v) > 1:
            ax_bar.text(bar.get_x() + bar.get_width()/2,
                        bar.get_y() + bar.get_height()/2,
                        f'{v:,.0f}', ha='center', va='center',
                        fontsize=9, color='white', fontweight='bold')
    bottoms += np.array(vals)

ax_bar.set_xticks(x)
ax_bar.set_xticklabels(strategies, fontsize=12)
ax_bar.set_ylabel('Revenue (EUR)', fontsize=11)
ax_bar.set_title('Total Revenue by Component', fontsize=13, fontweight='bold')
ax_bar.legend(loc='upper right', fontsize=10)
ax_bar.grid(True, alpha=0.25, axis='y')
for xi, total in zip(x, [opt_total_rev, bench_total_rev]):
    ax_bar.text(xi, total + abs(total)*0.015, f'Total: {total:,.2f} EUR',
                ha='center', va='bottom', fontsize=10, fontweight='bold')

# 2) Cumulative revenue lines
ax_cum_rev = fig.add_subplot(gs[1, 0])
ax_cum_rev.plot(results_df['PeriodStartLocal'], np.array(optimized_cumulative_revenue),
                color='darkgreen', linewidth=2, label='Optimised')
ax_cum_rev.plot(results_df['PeriodStartLocal'], np.array(benchmark_cumulative_revenue),
                color='darkorange', linewidth=2, linestyle='--', label='Benchmark')
ax_cum_rev.set_title('Cumulative Revenue Over Day', fontsize=12, fontweight='bold')
ax_cum_rev.set_ylabel('Cumulative Revenue (EUR)', fontsize=10)
ax_cum_rev.legend(fontsize=9)
ax_cum_rev.grid(True, alpha=0.3)
plt.setp(ax_cum_rev.xaxis.get_majorticklabels(), rotation=30, ha='right', fontsize=8)

# 3) Energy sold bar
ax_nrg = fig.add_subplot(gs[1, 1])
energy_vals = [opt_energy_mwh, bench_energy_mwh, actual_energy_mwh]
energy_labels = ['Optimised\nDA+IDA', 'Benchmark\nDA-only', 'Actual\nGeneration']
energy_colors = ['steelblue', 'darkorange', '#2ca02c']
bars = ax_nrg.bar(energy_labels, energy_vals, color=energy_colors, alpha=0.85, width=0.5)
for bar, v in zip(bars, energy_vals):
    ax_nrg.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002*max(energy_vals),
                f'{v:.2f} MWh', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax_nrg.set_title('Total Energy Sold on Market', fontsize=12, fontweight='bold')
ax_nrg.set_ylabel('Energy (MWh)', fontsize=10)
ax_nrg.grid(True, alpha=0.3, axis='y')
ax_nrg.set_ylim(0, max(energy_vals) * 1.12)

fig.suptitle('Revenue Conclusion: No-Curtailment Optimised vs Naive Benchmark',
             fontsize=15, fontweight='bold', y=1.01)

plt.show()

# ── Printed conclusion ───────────────────────────────────────────────────────
print()
print("╔══════════════════════════════════════════════════════════════════╗")
print("║       FINAL REVENUE CONCLUSION                                  ║")
print("║       No-Curtailment Optimised  vs  Naive Benchmark             ║")
print("╠══════════════════════════════════════════════════════════════════╣")
print(f"║  {'':63}║")
print(f"║  {'TOTAL REVENUE':63}║")
print(f"║  {'  No-Curtailment Optimised (DA + ID hedging):':<44} {opt_total_rev:>10,.2f} EUR  ║")
print(f"║  {'  Benchmark (Naive DA-only):':<44} {bench_total_rev:>10,.2f} EUR  ║")
print(f"║  {'  Difference:':<44} {diff_total:>+10,.2f} EUR  ║")
print(f"║  {'  Relative improvement:':<44} {diff_pct:>+10.2f} %    ║")
print(f"║  {'':63}║")
print(f"╠══════════════════════════════════════════════════════════════════╣")
print(f"║  {'REVENUE BREAKDOWN':63}║")
print(f"║  {'':10}{'Optimised':>18}  {'Benchmark':>18}  {'Diff':>10}  ║")
print(f"║  {'DA Revenue':<10} {opt_da_rev:>18,.2f}  {bench_da_rev:>18,.2f}  {opt_da_rev-bench_da_rev:>+10,.2f}  ║")
print(f"║  {'ID Revenue':<10} {opt_id_rev:>18,.2f}  {0.0:>18,.2f}  {opt_id_rev:>+10,.2f}  ║")
print(f"║  {'EEG Prem.':<10} {opt_eeg_rev:>18,.2f}  {bench_eeg_rev:>18,.2f}  {opt_eeg_rev-bench_eeg_rev:>+10,.2f}  ║")
print(f"║  {'Imbalance':<10} {opt_imb_rev:>18,.2f}  {bench_imb_rev:>18,.2f}  {opt_imb_rev-bench_imb_rev:>+10,.2f}  ║")
print(f"║  {'':63}║")
print(f"╠══════════════════════════════════════════════════════════════════╣")
print(f"║  {'ENERGY SOLD ON MARKET (MWh)':63}║")
print(f"║  {'  No-Curtailment Optimised (DA+IDA net):':<44} {opt_energy_mwh:>10,.2f} MWh  ║")
print(f"║  {'  Benchmark (DA committed):':<44} {bench_energy_mwh:>10,.2f} MWh  ║")
print(f"║  {'  Actual Generation delivered:':<44} {actual_energy_mwh:>10,.2f} MWh  ║")
print(f"║  {'':63}║")
print("╚══════════════════════════════════════════════════════════════════╝")

print()
print(summary.to_string(index=False))
